# σ-zero effectiveness — and tuning it for the detectors

**Part A** runs the σ-zero attack against DNN, AT-rFGSM, AT-MaxMA, DLA and PAD, with a learning-rate × threshold grid per model. **Part B** is the tuning notebook for the remaining detectors (KDE, DLA, DNN+, ICNN), including DLA's γ/α grid — the parameter search behind Part A's settings.

**Where this fits:** this is the culminating section — the organised version of everything before it. The defences come from `04-defenses/`, the attacks from `03-attacks/` (σ itself from `04-defenses/sigma_zero_attack.ipynb`), and the protocol is uniform: same data, same manipulation mask, same scoring, results saved as tensor archives for the comparison notebooks. The final published method lives in the separate sigma-binary repository.

*Consolidated from `Effectiveness_detectors_sigma_zero_attack.ipynb` and `parameter_tuning.ipynb` (originally in `thesis`). Outputs cleared.*

## Part A — The effectiveness runs

### Step 1 — Setup

# Loading data

In [ ]:
import torch
import torch.nn as nn

# Assuming you are using a GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

In [ ]:
import gdown

download_links = [
                  'https://drive.google.com/uc?id=1dkoFDyJMuGewWX-LE5rb7zkRU6XkSa2z',
                  'https://drive.google.com/uc?id=1oHJBbqyKybmlFFRZ7-uftn77Vg0Hr_A5',
                  'https://drive.google.com/uc?id=1YwxR5CCNnG5gjeFBlBA8mEfDQM22mLj_',
                  'https://drive.google.com/uc?id=1XB5jdefjnGSxrb27SO-65JZpz5UcsNh2',
                  'https://drive.google.com/uc?id=1Ahw4prIhm322LU53Yhkju5ixd6q1KCJ6',
                  'https://drive.google.com/uc?id=18_YhP6qHHmu52CfANYD2LqxrOzq5ECxl',
                  'https://drive.google.com/uc?id=140wKHyRpKurSuvEOswKdyH3kKRPg8VW_',
                  'https://drive.google.com/uc?id=1pTokv8xKvZoUo8kejJc7N757iNfHmt2s',
                  'https://drive.google.com/uc?id=107sDEWVIJZJWC1SIWASEepdRXB3VWlCZ',
                  'https://drive.google.com/uc?id=1sFBZ3A8VQ-iDjmwASg4u5uHQQR8kVJy9',


]


output_filepath = '/content/'
for link in download_links:
  gdown.download(link, output_filepath)

In [ ]:
import zipfile

# Define the path to the zip file and the extraction path
zip_file_path = '/content/datasets.zip'
extract_to_path = '/content/'

# Unzip the file
with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
    zip_ref.extractall(extract_to_path)

In [ ]:
import torch
from torch.utils.data import DataLoader, TensorDataset

# Load tensors
train_features = torch.load('train_features.pt', weights_only=True).float()
train_labels = torch.load('train_labels.pt', weights_only=True).long()
val_features = torch.load('val_features.pt', weights_only=True).float()
val_labels = torch.load('val_labels.pt', weights_only=True).long()
test_features = torch.load('test_features.pt', weights_only=True).float()
test_labels = torch.load('test_labels.pt', weights_only=True).long()

# Print data shapes for verification
print(f"Train Features Shape: {train_features.shape}")
print(f"Train Labels Shape: {train_labels.shape}")
print(f"Validation Features Shape: {val_features.shape}")
print(f"Validation Labels Shape: {val_labels.shape}")
print(f"Test Features Shape: {test_features.shape}")
print(f"Test Labels Shape: {test_labels.shape}")

# Create datasets
train_dataset = TensorDataset(train_features, train_labels)
val_dataset = TensorDataset(val_features, val_labels)
test_dataset = TensorDataset(test_features, test_labels)

# Set batch size
batch_size = 10500
# Create DataLoaders
train_loader = DataLoader(dataset=train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(dataset=val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(dataset=test_dataset, batch_size=batch_size, shuffle=False)

# Optional: Verify DataLoader sizes
print('----------------------------------------------')
print(f"Train Loader Batch Count: {len(train_loader)}")
print(f"Validation Loader Batch Count: {len(val_loader)}")
print(f"Test Loader Batch Count: {len(test_loader)}")

In [ ]:
# Function to count samples for each label
def count_labels(labels):
    count_0 = (labels == 0).sum().item()  # Count samples with label = 0
    count_1 = (labels == 1).sum().item()  # Count samples with label = 1
    return count_0, count_1

# Count labels for each dataset
train_count_0, train_count_1 = count_labels(train_labels)
val_count_0, val_count_1 = count_labels(val_labels)
test_count_0, test_count_1 = count_labels(test_labels)

# Calculate total counts across all datasets
total_count_0 = train_count_0 + val_count_0 + test_count_0
total_count_1 = train_count_1 + val_count_1 + test_count_1

# Print results
print(f"Training set             \t Label 0: {train_count_0}\t Label 1: {train_count_1}")
print(f"Validation set           \t Label 0: {val_count_0}\t Label 1: {val_count_1}")
print(f"Test set                 \t Label 0: {test_count_0}\t Label 1: {test_count_1}")
print(f"Total across all datasets\t Label 0: {total_count_0}\t Label 1: {total_count_1}")

In [ ]:
print(f"The avrage number of feature in Training set: {train_features.sum()/train_features.shape[0]:.2f}")
print(f"The avrage number of feature in Validation set: {val_features.sum()/val_features.shape[0]:.2f}")
print(f"The avrage number of feature in Test set: {test_features.sum()/test_features.shape[0]:.2f}")

In [ ]:
feature_mask = torch.load('manipulation_x.pt', weights_only=True, map_location=device)
feature_mask.sum()

# Models

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

def to_tensor(feature_x=None, labels=None, device='cpu'):

    def _to_torch_tensor(mat):
        mat = torch.DoubleTensor(mat)
        return mat

    feature_x = _to_torch_tensor(feature_x).to(device)
    if labels is None:
        return feature_x
    else:
        labels = torch.LongTensor(labels).to(device)
        return feature_x, labels

def to_device(feature_x=None, labels=None, device='cpu'):
    if feature_x is not None:
        assert isinstance(feature_x, torch.Tensor)
        feature_x = feature_x.to(device)
    if labels is not None:
        assert isinstance(labels, torch.Tensor)
        labels = labels.to(device)
    return feature_x, labels

class MalwareDetectionDNN(nn.Module):
    def __init__(self, input_size, n_classes, device='cpu', name='DNN', dropout = 0.6, alpha = 0.2, smooth = False):
        """
        Construct malware detector

        Parameters
        ----------
        @param input_size: Integer, the dimentionality number of input vector
        @param n_classes: Integer, the number of classes, n=2
        @param device: String, 'cpu' or 'cuda'
        @param name: String, model name
        """
        super(MalwareDetectionDNN, self).__init__()
        self.input_size = input_size
        self.n_classes = n_classes
        self.device = device
        self.name = name
        self.dense_hidden_units = [200, 200]
        self.dropout = dropout
        self.alpha_ = alpha
        self.smooth = smooth


        self.dense_layers = []
        if len(self.dense_hidden_units) >= 1:
            self.dense_layers.append(nn.Linear(self.input_size, self.dense_hidden_units[0]))
        else:
            raise ValueError("Expect at least one hidden layer.")
        for i in range(len(self.dense_hidden_units[0:-1])):
            self.dense_layers.append(nn.Linear(self.dense_hidden_units[i],  # start from idx=1
                                               self.dense_hidden_units[i + 1]))
        self.dense_layers.append(nn.Linear(self.dense_hidden_units[-1], self.n_classes))
        # registration
        for idx_i, dense_layer in enumerate(self.dense_layers):
            self.add_module('nn_model_layer_{}'.format(idx_i), dense_layer)

        if self.smooth:
            self.activation_func = F.selu  # partial(F.elu, alpha=self.alpha_)
        else:
            self.activation_func = F.relu


    def forward(self, x):
        """
        Go through the neural network

        Parameters
        ----------
        @param x: 2D tensor, feature representation
        """
        for dense_layer in self.dense_layers[:-1]:
            x = self.activation_func(dense_layer(x))

        latent_representation = F.dropout(x, self.dropout, training=self.training)
        logits = self.dense_layers[-1](latent_representation)
        return logits

    def inference(self, test_data_producer):
        confidences = []
        gt_labels = []
        self.eval()
        with torch.no_grad():
            for x, y in test_data_producer:
                x, y = to_device(x.double(), y.long(), self.device)
                logits = self.forward(x)
                confidences.append(F.softmax(logits, dim=-1))
                gt_labels.append(y)
        confidences = torch.vstack(confidences)
        gt_labels = torch.cat(gt_labels, dim=0)
        return confidences, gt_labels


    def inference_batch_wise(self, x):
        """
        support malware samples solely
        """
        assert isinstance(x, torch.Tensor)
        logit = self.forward(x)
        return torch.softmax(logit, dim=-1).detach().cpu().numpy(), np.ones((logit.size()[0],))

    def predict(self, test_data_producer, indicator_masking=True):
        """
        predict labels and conduct evaluation

        Parameters
        --------
        @param test_data_producer, torch.DataLoader
        """
        # evaluation
        confidence, y_true = self.inference(test_data_producer)
        y_pred = confidence.argmax(1).cpu().numpy()
        y_true = y_true.cpu().numpy()

        def measurement(_y_true, _y_pred):
            from sklearn.metrics import f1_score, accuracy_score, confusion_matrix, balanced_accuracy_score
            accuracy = accuracy_score(_y_true, _y_pred)
            b_accuracy = balanced_accuracy_score(_y_true, _y_pred)
            print(f"The accuracy on the test dataset is {(accuracy * 100):.2f}%")
            print(f"The balanced accuracy on the test dataset is {(b_accuracy * 100):.2f}%")

            if np.any([np.all(_y_true == i) for i in range(self.n_classes)]):
                return

            tn, fp, fn, tp = confusion_matrix(_y_true, _y_pred).ravel()
            fpr = fp / float(tn + fp)
            fnr = fn / float(tp + fn)
            f1 = f1_score(_y_true, _y_pred, average='binary')
            #print("Other evaluation metrics we may need:")
            print(f"False Negative Rate (FNR) is {fnr * 100:.2f}%, False Positive Rate (FPR) is {fpr * 100:.2f}%, F1 score is {f1 * 100:.2f}%")

        measurement(y_true, y_pred)


    def load(self, model_save_path):
        """
        load model parameters from disk
        """
        self.load_state_dict(torch.load(model_save_path, map_location=torch.device(device)))

    # loading rFGSM
    def load2(self, model_save_path):
        ckpt = torch.load(model_save_path, map_location=torch.device(device))
        self.load_state_dict(ckpt['model'])

In [ ]:
import numpy as np

class DetectorTemplate(object):
    def __init__(self):
        self.tau = None
        self.is_detector_enabled = True

    def forward(self, x):
        """class prediction & density estimation"""
        raise NotImplementedError

    def get_threshold(self):
        """calculate the threshold for rejecting outliers"""
        raise NotImplementedError

    def get_tau_sample_wise(self):
        raise NotImplementedError

    def indicator(self):
        """return a vector of boolean flags indicating whether reject a sample"""
        raise NotImplementedError


def to_tensor(feature_x=None, labels=None, device='cpu'):

    def _to_torch_tensor(mat):
        mat = torch.DoubleTensor(mat)
        return mat

    feature_x = _to_torch_tensor(feature_x).to(device)
    if labels is None:
        return feature_x
    else:
        labels = torch.LongTensor(labels).to(device)
        return feature_x, labels

def to_device(feature_x=None, labels=None, device='cpu'):
    if feature_x is not None:
        assert isinstance(feature_x, torch.Tensor)
        feature_x = feature_x.to(device)
    if labels is not None:
        assert isinstance(labels, torch.Tensor)
        labels = labels.to(device)
    return feature_x, labels


class KernelDensityEstimation(nn.Module, DetectorTemplate):
    """
    kernel density estimation upon the penultimate layer

    parameters
    -------------
    @param model, torch.nn.Module, an instantiation of model object
    @param bandwidth, float, variance of Gaussian density function
    @param n_classes, integer, number of classes
    @param ratio, float [0,1], ratio for computing the threshold
    """

    def __init__(self, model, n_centers=1000, bandwidth=20., n_classes=2, ratio=0.95, name='KDE', device='cpu'):
        super(KernelDensityEstimation, self).__init__()
        assert isinstance(model, torch.nn.Module)
        self.model = model
        self.device = device
        self.name = name
        self.n_centers = n_centers
        self.bandwidth = bandwidth
        self.n_classes = n_classes
        self.ratio = ratio
        self.gaussian_means = None

        self.tau = nn.Parameter(torch.zeros([self.n_classes, ], device=self.device), requires_grad=False)

    def forward(self, x):
        for dense_layer in self.model.dense_layers[:-1]:
            x = self.model.activation_func(dense_layer(x))
        logits = self.model.dense_layers[-1](x)
        x_prob = self.forward_g(x, logits.argmax(1).detach())
        return logits, x_prob

    def forward_f(self, x):
        for dense_layer in self.model.dense_layers[:-1]:
            x = self.model.activation_func(dense_layer(x))
        logits = self.model.dense_layers[-1](x)
        return logits, x

    def forward_g(self, x_hidden, y_pred):
        """
        1 / |X_| * \sum(exp(||x - x_||_2^2/sigma^2))

        parameters
        -----------
        @param x_hidden, torch.tensor, node hidden representation
        @param y_pred, torch.tensor, prediction
        """
        size = x_hidden.size()[0]
        dist = [torch.sum(torch.square(means.unsqueeze(dim=0) - x_hidden.unsqueeze(dim=1)), dim=-1) for means in
                self.gaussian_means]
        kd = torch.stack([torch.mean(torch.exp(-d / self.bandwidth ** 2), dim=-1) for d in dist], dim=1)
        # return p(x|y=y_pred)
        return -1 * kd[torch.arange(size), y_pred]

    def get_threshold(self, validation_data_producer, ratio=None):
        """
        get the threshold for density estimation
        :@param validation_data_producer: Object, an iterator for producing validation dataset
        """
        ratio = ratio if ratio is not None else self.ratio
        assert 0 <= ratio <= 1
        self.eval()
        probabilities = []
        gt_labels = []
        with torch.no_grad():
            for x_val, y_val in validation_data_producer:
                x_val, y_val = to_tensor(x_val.double(), y_val.long(), self.device)
                logits, x_prob = self.forward(x_val)
                probabilities.append(x_prob)
                gt_labels.append(y_val)
            prob = torch.cat(probabilities, dim=0)
            gt_labels = torch.cat(gt_labels)
            for i in range(self.n_classes):
                prob_x_y = prob[gt_labels == i]
                s, _ = torch.sort(prob_x_y)
                self.tau[i] = s[int((s.shape[0] - 1) * ratio)]

    def predict(self, test_data_producer, indicator_masking=True):
        # evaluation on detector & indicator
        y_cent, x_prob, y_true = self.inference(test_data_producer)
        y_pred = y_cent.argmax(1).cpu().numpy()
        y_true = y_true.cpu().numpy()
        indicator_flag = self.indicator(x_prob, y_pred).cpu().numpy()

        def measurement(_y_true, _y_pred):
            from sklearn.metrics import f1_score, accuracy_score, confusion_matrix, balanced_accuracy_score
            accuracy = accuracy_score(_y_true, _y_pred)
            b_accuracy = balanced_accuracy_score(_y_true, _y_pred)
            print(f"The accuracy on the test dataset is {(accuracy * 100):.2f}%")
            print(f"The balanced accuracy on the test dataset is {(b_accuracy * 100):.2f}%")

            if np.any([np.all(_y_true == i) for i in range(self.n_classes)]):
                return

            tn, fp, fn, tp = confusion_matrix(_y_true, _y_pred).ravel()
            fpr = fp / float(tn + fp)
            fnr = fn / float(tp + fn)
            f1 = f1_score(_y_true, _y_pred, average='binary')
            #print("Other evaluation metrics we may need:")
            print(f"False Negative Rate (FNR) is {fnr * 100:.2f}%, False Positive Rate (FPR) is {fpr * 100:.2f}%, F1 score is {f1 * 100:.2f}%")



        if indicator_masking:
            flag_of_retaining = indicator_flag  # excluding the examples with ``not sure'' response
            y_pred = y_pred[flag_of_retaining]
            y_true = y_true[flag_of_retaining]
            ExS = (len(y_cent) - len(y_true))/len(y_cent)
            print('Deferred option : ')
            print('The threshold is {}'.format(self.tau))
            print(f"Percentage of excluded samples is {ExS * 100:.2f}%")
            measurement(y_true, y_pred)

        else:
            # instead filtering out examples, here resets the prediction as 1 (test model on perturbed examples)
            y_pred[~indicator_flag] = 1.
            print('Conservative option : ')
            print('The threshold is {}'.format(self.tau))
            measurement(y_true, y_pred)





    def eval(self):
        self.model.eval()

    def inference(self, test_data_producer):
        y_cent, x_prob = [], []
        gt_labels = []
        self.eval()
        with torch.no_grad():
            for x, y in test_data_producer:
                x, y = x.to(device), y.to(device)
                logits_f, logits_g = self.forward(x)
                y_cent.append(F.softmax(logits_f, dim=-1))
                x_prob.append(logits_g)
                gt_labels.append(y)

        gt_labels = torch.cat(gt_labels, dim=0)
        y_cent = torch.cat(y_cent, dim=0)
        x_prob = torch.cat(x_prob, dim=0)
        return y_cent, x_prob, gt_labels

    def inference_batch_wise(self, x):
        assert isinstance(x, torch.Tensor)

        logits, x_prob = self.forward(x)
        return torch.softmax(logits, dim=-1).detach().cpu().numpy(), x_prob.detach().cpu().numpy()

    def get_tau_sample_wise(self, y_pred=None):
        return self.tau[y_pred]

    def indicator(self, x_prob, y_pred=None):
        assert y_pred is not None
        if isinstance(x_prob, np.ndarray):
            x_prob = torch.tensor(x_prob, device=self.device)
            return (x_prob <= self.get_tau_sample_wise(y_pred)).cpu().numpy()
        elif isinstance(x_prob, torch.Tensor):
            return x_prob <= self.get_tau_sample_wise(y_pred)
        else:
            raise TypeError("Tensor or numpy.ndarray are expected.")
        # res = probability.reshape(-1, 1).repeat_interleave(2, dim=1) >= self.tau
        # return res[torch.arange(res.size()[0]), y_pred]


    def load(self, model_save_path):
        ckpt = torch.load(model_save_path, map_location=torch.device(device))
        self.gaussian_means = ckpt['gaussian_means']
        self.tau = ckpt['tau']
        self.model.load_state_dict(ckpt['base_model'])

In [ ]:
import time
import warnings
import os.path as path
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F


import numpy as np


class DetectorTemplate(object):
    def __init__(self):
        self.tau = None
        self.is_detector_enabled = True

    def forward(self, x):
        """class prediction & density estimation"""
        raise NotImplementedError

    def get_threshold(self):
        """calculate the threshold for rejecting outliers"""
        raise NotImplementedError

    def get_tau_sample_wise(self):
        raise NotImplementedError

    def indicator(self):
        """return a vector of boolean flags indicating whether reject a sample"""
        raise NotImplementedError



def to_tensor(feature_x=None, labels=None, device='cpu'):

    def _to_torch_tensor(mat):
        mat = torch.DoubleTensor(mat)
        return mat

    feature_x = _to_torch_tensor(feature_x).to(device)
    if labels is None:
        return feature_x
    else:
        labels = torch.LongTensor(labels).to(device)
        return feature_x, labels

def to_device(feature_x=None, labels=None, device='cpu'):
    if feature_x is not None:
        assert isinstance(feature_x, torch.Tensor)
        feature_x = feature_x.to(device)
    if labels is not None:
        assert isinstance(labels, torch.Tensor)
        labels = labels.to(device)
    return feature_x, labels



class AMalwareDetectionDLA(nn.Module, DetectorTemplate):
    def __init__(self, md_nn_model, input_size, n_classes, ratio=0.95,
                 device='cpu', name='', dropout = 0.6, alpha = 0.2, smooth = False):
        nn.Module.__init__(self)
        DetectorTemplate.__init__(self)
        self.input_size = input_size
        self.n_classes = n_classes
        self.ratio = ratio
        self.device = device
        self.name = name
        self.dense_hidden_units = [200, 200]
        self.dropout = dropout
        self.alpha_ = alpha
        self.smooth = smooth


        # malware detector
        if md_nn_model is not None and isinstance(md_nn_model, nn.Module):
            self.md_nn_model = md_nn_model
            self.is_fitting_md_model = False  # the model is trained by default
        else:
            self.md_nn_model = MalwareDetectionDNN(self.input_size,
                                                   n_classes,
                                                   self.device,
                                                   name)
            self.is_fitting_md_model = True

        assert len(self.dense_hidden_units) >= 1, "Expected at least one hidden layer."
        self.alarm_nn_model = TorchAlarm(input_size=sum(self.md_nn_model.dense_hidden_units))

        self.tau = nn.Parameter(torch.zeros([1, ], device=self.device), requires_grad=False)

    def forward(self, x):
        hidden_activations = []
        for dense_layer in self.md_nn_model.dense_layers[:-1]:
            x = self.md_nn_model.activation_func(dense_layer(x))
            hidden_activations.append(x)
        logits = self.md_nn_model.dense_layers[-1](x)
        hidden_activations = torch.cat(hidden_activations, dim=-1)
        x_prob = self.alarm_nn_model(hidden_activations).reshape(-1)
        return logits, x_prob

    def predict(self, test_data_producer, indicator_masking=True):
        """
        predict labels and conduct evaluation on detector & indicator

        Parameters
        --------
        @param test_data_producer, torch.DataLoader
        @param indicator_masking, whether filtering out the examples with low density or masking their values
        """
        y_cent, x_prob, y_true = self.inference(test_data_producer)
        y_pred = y_cent.argmax(1).cpu().numpy()
        y_true = y_true.cpu().numpy()
        indicator_flag = self.indicator(x_prob).cpu().numpy()

        def measurement(_y_true, _y_pred):
            from sklearn.metrics import f1_score, accuracy_score, confusion_matrix, balanced_accuracy_score
            accuracy = accuracy_score(_y_true, _y_pred)
            b_accuracy = balanced_accuracy_score(_y_true, _y_pred)
            print(f"The accuracy on the test dataset is {(accuracy * 100):.2f}%")
            print(f"The balanced accuracy on the test dataset is {(b_accuracy * 100):.2f}%")

            if np.any([np.all(_y_true == i) for i in range(self.n_classes)]):
                return

            tn, fp, fn, tp = confusion_matrix(_y_true, _y_pred).ravel()
            fpr = fp / float(tn + fp)
            fnr = fn / float(tp + fn)
            f1 = f1_score(_y_true, _y_pred, average='binary')
            #print("Other evaluation metrics we may need:")
            print(f"False Negative Rate (FNR) is {fnr * 100:.2f}%, False Positive Rate (FPR) is {fpr * 100:.2f}%, F1 score is {f1 * 100:.2f}%")



        if indicator_masking:
            flag_of_retaining = indicator_flag  # excluding the examples with ``not sure'' response
            y_pred = y_pred[flag_of_retaining]
            y_true = y_true[flag_of_retaining]
            ExS = (len(y_cent) - len(y_true))/len(y_cent)
            print('Deferred option : ')
            print('The threshold is {:.5}'.format(self.tau.item()))
            print(f"Percentage of excluded samples is {ExS * 100:.2f}%")
            measurement(y_true, y_pred)

        else:
            # instead filtering out examples, here resets the prediction as 1 (test model on perturbed examples)
            y_pred[~indicator_flag] = 1.
            print('Conservative option : ')
            print('The threshold is {:.5}'.format(self.tau.item()))
            measurement(y_true, y_pred)


    def inference(self, test_data_producer):
        y_cent, x_prob = [], []
        gt_labels = []
        self.eval()
        with torch.no_grad():
            for x, y in test_data_producer:
                x, y = to_device(x.double(), y.long(), self.device)
                logits, x_logits = self.forward(x)
                y_cent.append(F.softmax(logits, dim=-1))
                x_prob.append(x_logits)
                gt_labels.append(y)

        gt_labels = torch.cat(gt_labels, dim=0)
        y_cent = torch.cat(y_cent, dim=0)
        x_prob = torch.cat(x_prob, dim=0)
        return y_cent, x_prob, gt_labels

    def inference_batch_wise(self, x):
        assert isinstance(x, torch.Tensor)
        self.eval()
        logits, x_prob = self.forward(x)
        return torch.softmax(logits, dim=-1).detach().cpu().numpy(), x_prob.detach().cpu().numpy()

    def get_tau_sample_wise(self, y_pred=None):
        return self.tau

    def indicator(self, x_prob, y_pred=None):
        """
        Return 'True' if a sample is original, and otherwise 'False' is returned.
        """
        if isinstance(x_prob, np.ndarray):
            x_prob = torch.tensor(x_prob, device=self.device)
            return (x_prob <= self.tau).cpu().numpy()
        elif isinstance(x_prob, torch.Tensor):
            return x_prob <= self.tau
        else:
            raise TypeError("Tensor or numpy.ndarray are expected.")

    def get_threshold(self, validation_data_producer, ratio=None):
        """
        get the threshold for adversary detection
        :@param validation_data_producer: Object, an iterator for producing validation dataset
        """
        self.eval()
        ratio = ratio if ratio is not None else self.ratio
        assert 0 <= ratio <= 1
        probabilities = []
        with torch.no_grad():
            for x_val, y_val in validation_data_producer:
                x_val, y_val = to_tensor(x_val.double(), y_val.long(), self.device)
                _1, x_prob = self.forward(x_val)
                probabilities.append(x_prob)
            s, _ = torch.sort(torch.cat(probabilities, dim=0))
            i = int((s.shape[0] - 1) * ratio)
            assert i >= 0
            self.tau[0] = s[i]

    def load(self, model_save_path):
        self.load_state_dict(torch.load(model_save_path, map_location=torch.device(device)))


class TorchAlarm(torch.nn.Module):
    def __init__(self, input_size):
        super().__init__()

        self.layers = torch.nn.ModuleList([
            torch.nn.Linear(input_size, 112),
            torch.nn.ReLU(),
            torch.nn.Linear(112, 100),
            torch.nn.ReLU(),
            torch.nn.Linear(100, 300),
            torch.nn.ReLU(),
            torch.nn.Linear(300, 200),
            torch.nn.ReLU(),
            torch.nn.Linear(200, 77),
            torch.nn.ReLU(),
            torch.nn.Linear(77, 1),
        ])

    def __call__(self, x, training=False):
        if not isinstance(x, torch.Tensor):
            x = torch.tensor(x, dtype=torch.float32)
        for layer in self.layers:
            x = layer(x)
        return x

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import numpy as np

class DetectorTemplate(object):
    def __init__(self):
        self.tau = None
        self.is_detector_enabled = True

    def forward(self, x):
        """class prediction & density estimation"""
        raise NotImplementedError

    def get_threshold(self):
        """calculate the threshold for rejecting outliers"""
        raise NotImplementedError

    def get_tau_sample_wise(self):
        raise NotImplementedError

    def indicator(self):
        """return a vector of boolean flags indicating whether reject a sample"""
        raise NotImplementedError



def to_tensor(feature_x=None, labels=None, device='cpu'):

    def _to_torch_tensor(mat):
        mat = torch.DoubleTensor(mat)
        return mat

    feature_x = _to_torch_tensor(feature_x).to(device)
    if labels is None:
        return feature_x
    else:
        labels = torch.LongTensor(labels).to(device)
        return feature_x, labels

def to_device(feature_x=None, labels=None, device='cpu'):
    if feature_x is not None:
        assert isinstance(feature_x, torch.Tensor)
        feature_x = feature_x.to(device)
    if labels is not None:
        assert isinstance(labels, torch.Tensor)
        labels = labels.to(device)
    return feature_x, labels







class AMalwareDetectionDNNPlus(nn.Module, DetectorTemplate):
    def __init__(self, md_nn_model, input_size, n_classes, ratio=0.95,
                 device='cpu', name='', dropout = 0.6, alpha = 0.2, smooth = False):
        nn.Module.__init__(self)
        DetectorTemplate.__init__(self)
        self.input_size = input_size
        self.n_classes = n_classes
        self.ratio = ratio
        self.device = device
        self.name = name
        self.dense_hidden_units = [200, 200]
        self.dropout = dropout
        self.alpha_ = alpha
        self.smooth = smooth

        # malware detector
        if md_nn_model is not None and isinstance(md_nn_model, nn.Module):
            self.md_nn_model = md_nn_model
            self.is_fitting_md_model = False  # the model is trained by default
        else:
            self.md_nn_model = MalwareDetectionDNN(self.input_size,
                                                   self.n_classes,
                                                   self.device,
                                                   name)
            self.is_fitting_md_model = True

        self.amd_nn_plus = MalwareDetectionDNN(self.input_size,
                                               self.n_classes + 1,
                                               self.device,
                                               name)

        self.tau = nn.Parameter(torch.zeros([1, ], device=self.device), requires_grad=False)


    def forward(self, x):
        logits = self.amd_nn_plus(x)
        logits -= torch.amax(logits, dim=-1, keepdim=True).detach()  # increasing the stability, which might be helpful
        return logits, torch.softmax(logits, dim=-1)[:, -1]

    def predict(self, test_data_producer, indicator_masking=True):
        """
        predict labels and conduct evaluation on detector & indicator

        Parameters
        --------
        @param test_data_producer, torch.DataLoader
        @param indicator_masking, whether filtering out the examples with low density or masking their values
        """
        y_cent, x_prob, y_true = self.inference(test_data_producer)
        y_pred = y_cent.argmax(1).cpu().numpy()
        y_true = y_true.cpu().numpy()
        indicator_flag = self.indicator(x_prob).cpu().numpy()

        def measurement(_y_true, _y_pred):
            from sklearn.metrics import f1_score, accuracy_score, confusion_matrix, balanced_accuracy_score
            accuracy = accuracy_score(_y_true, _y_pred)
            b_accuracy = balanced_accuracy_score(_y_true, _y_pred)
            print(f"The accuracy on the test dataset is {(accuracy * 100):.2f}%")
            print(f"The balanced accuracy on the test dataset is {(b_accuracy * 100):.2f}%")

            if np.any([np.all(_y_true == i) for i in range(self.n_classes)]):
                return

            tn, fp, fn, tp = confusion_matrix(_y_true, _y_pred).ravel()
            fpr = fp / float(tn + fp)
            fnr = fn / float(tp + fn)
            f1 = f1_score(_y_true, _y_pred, average='binary')
            #print("Other evaluation metrics we may need:")
            print(f"False Negative Rate (FNR) is {fnr * 100:.2f}%, False Positive Rate (FPR) is {fpr * 100:.2f}%, F1 score is {f1 * 100:.2f}%")



        if indicator_masking:
            flag_of_retaining = indicator_flag  # excluding the examples with ``not sure'' response
            y_pred = y_pred[flag_of_retaining]
            y_true = y_true[flag_of_retaining]
            ExS = (len(y_cent) - len(y_true))/len(y_cent)
            print('Deferred option : ')
            print('The threshold is {:.5}'.format(self.tau.item()))
            print(f"Percentage of excluded samples is {ExS * 100:.2f}%")
            measurement(y_true, y_pred)

        else:
            # instead filtering out examples, here resets the prediction as 1 (test model on perturbed examples)
            y_pred[~indicator_flag] = 1.
            print('Conservative option : ')
            print('The threshold is {:.5}'.format(self.tau.item()))
            measurement(y_true, y_pred)



    def inference(self, test_data_producer):
        y_cent, x_prob = [], []
        gt_labels = []
        self.eval()
        with torch.no_grad():
            for x, y in test_data_producer:
                x, y = to_device(x.double(), y.long(), self.device)
                logits, x_cent = self.forward(x)
                y_cent.append(F.softmax(logits, dim=-1)[:, :2])
                x_prob.append(x_cent)
                gt_labels.append(y)

        gt_labels = torch.cat(gt_labels, dim=0)
        y_cent = torch.cat(y_cent, dim=0)
        x_prob = torch.cat(x_prob, dim=0)
        return y_cent, x_prob, gt_labels

    def inference_batch_wise(self, x):
        assert isinstance(x, torch.Tensor)
        self.eval()
        logits, g = self.forward(x)
        x_cent = torch.softmax(logits, dim=-1).detach().cpu().numpy()[:, :2]

        return x_cent, g.detach().cpu().numpy()

    def get_tau_sample_wise(self, y_pred=None):
        return self.tau

    def indicator(self, x_prob, y_pred=None):
        """
        Return 'True' if a sample is original, and otherwise 'False' is returned.
        """
        if isinstance(x_prob, np.ndarray):
            x_prob = torch.tensor(x_prob, device=self.device)
            return (x_prob <= self.tau).cpu().numpy()
        elif isinstance(x_prob, torch.Tensor):
            return x_prob <= self.tau
        else:
            raise TypeError("Tensor or numpy.ndarray are expected.")

    def get_threshold(self, validation_data_producer, ratio=None):
        """
        get the threshold for adversary detection
        :@param validation_data_producer: Object, an iterator for producing validation dataset
        """
        self.eval()
        ratio = ratio if ratio is not None else self.ratio
        assert 0 <= ratio <= 1
        probabilities = []
        with torch.no_grad():
            for x_val, y_val in validation_data_producer:
                x_val, y_val = to_tensor(x_val.double(), y_val.long(), self.device)
                _1, x_cent = self.forward(x_val)
                probabilities.append(x_cent)
            s, _ = torch.sort(torch.cat(probabilities, dim=0))
            i = int((s.shape[0] - 1) * ratio)
            assert i >= 0
            self.tau[0] = s[i]

    def load(self, model_save_path):
        self.load_state_dict(torch.load(model_save_path, map_location=torch.device(device)))

In [ ]:
import time
import warnings
import os.path as path
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import numpy as np


class DetectorTemplate(object):
    def __init__(self):
        self.tau = None
        self.is_detector_enabled = True

    def forward(self, x):
        """class prediction & density estimation"""
        raise NotImplementedError

    def get_threshold(self):
        """calculate the threshold for rejecting outliers"""
        raise NotImplementedError

    def get_tau_sample_wise(self):
        raise NotImplementedError

    def indicator(self):
        """return a vector of boolean flags indicating whether reject a sample"""
        raise NotImplementedError


def to_tensor(feature_x=None, labels=None, device='cpu'):

    def _to_torch_tensor(mat):
        mat = torch.DoubleTensor(mat)
        return mat

    feature_x = _to_torch_tensor(feature_x).to(device)
    if labels is None:
        return feature_x
    else:
        labels = torch.LongTensor(labels).to(device)
        return feature_x, labels

def to_device(feature_x=None, labels=None, device='cpu'):
    if feature_x is not None:
        assert isinstance(feature_x, torch.Tensor)
        feature_x = feature_x.to(device)
    if labels is not None:
        assert isinstance(labels, torch.Tensor)
        labels = labels.to(device)
    return feature_x, labels


class AdvMalwareDetectorICNN(nn.Module, DetectorTemplate):
    def __init__(self, md_nn_model, input_size, n_classes, ratio=0.95,
                 device='cpu', name='ICNN', ):
        nn.Module.__init__(self)
        DetectorTemplate.__init__(self)
        self.md_nn_model = md_nn_model
        self.input_size = input_size
        self.n_classes = n_classes
        self.ratio = ratio
        self.device = device
        self.name = name
        self.dense_hidden_units = [200, 200]


        if hasattr(self.md_nn_model, 'smooth'):
            if not self.md_nn_model.smooth:  # non-smooth, exchange it
                for name, child in self.md_nn_model.named_children():
                    if isinstance(child, nn.ReLU):
                        self.md_nn_model._modules['relu'] = nn.SELU()
        else:
            for name, child in self.md_nn_model.named_children():
                if isinstance(child, nn.ReLU):
                    self.md_nn_model._modules['relu'] = nn.SELU()
        self.md_nn_model = self.md_nn_model.to(self.device)

        # input convex neural network
        self.non_neg_dense_layers = []
        if len(self.dense_hidden_units) < 1:
            raise ValueError("Expect at least one hidden layer.")
        for i in range(len(self.dense_hidden_units[0:-1])):
            self.non_neg_dense_layers.append(nn.Linear(self.dense_hidden_units[i],  # start from idx=1
                                                       self.dense_hidden_units[i + 1],
                                                       bias=False))
        self.non_neg_dense_layers.append(nn.Linear(self.dense_hidden_units[-1], 1, bias=False))
        # registration
        for idx_i, dense_layer in enumerate(self.non_neg_dense_layers):
            self.add_module('non_neg_layer_{}'.format(idx_i), dense_layer)

        self.dense_layers = []
        self.dense_layers.append(nn.Linear(self.input_size, self.dense_hidden_units[0]))
        for i in range(len(self.dense_hidden_units[1:])):
            self.dense_layers.append(nn.Linear(self.input_size, self.dense_hidden_units[i]))
        self.dense_layers.append(nn.Linear(self.input_size, 1))
        # registration
        for idx_i, dense_layer in enumerate(self.dense_layers):
            self.add_module('layer_{}'.format(idx_i), dense_layer)

        self.tau = nn.Parameter(torch.zeros([1, ], device=self.device), requires_grad=False)


    def forward_f(self, x):
        return self.md_nn_model(x)

    def forward_g(self, x):
        prev_x = None
        for i, dense_layer in enumerate(self.dense_layers):
            x_add = []
            x1 = dense_layer(x)
            x_add.append(x1)
            if prev_x is not None:
                x2 = self.non_neg_dense_layers[i - 1](prev_x)
                x_add.append(x2)
            prev_x = torch.sum(torch.stack(x_add, dim=0), dim=0)
            if i < len(self.dense_layers):
                prev_x = F.selu(prev_x)
        return prev_x.reshape(-1)

    def forward(self, x):
        return self.forward_f(x), self.forward_g(x)

    def predict(self, test_data_producer, indicator_masking=True):
        """
        predict labels and conduct evaluation on detector & indicator

        Parameters
        --------
        @param test_data_producer, torch.DataLoader
        @param indicator_masking, whether filtering out the examples with low density or masking their values
        """
        y_cent, x_prob, y_true = self.inference(test_data_producer)
        y_pred = y_cent.argmax(1).cpu().numpy()
        y_true = y_true.cpu().numpy()
        indicator_flag = self.indicator(x_prob).cpu().numpy()

        def measurement(_y_true, _y_pred):
            from sklearn.metrics import f1_score, accuracy_score, confusion_matrix, balanced_accuracy_score
            accuracy = accuracy_score(_y_true, _y_pred)
            b_accuracy = balanced_accuracy_score(_y_true, _y_pred)
            print(f"The accuracy on the test dataset is {(accuracy * 100):.2f}%")
            print(f"The balanced accuracy on the test dataset is {(b_accuracy * 100):.2f}%")

            if np.any([np.all(_y_true == i) for i in range(self.n_classes)]):
                return

            tn, fp, fn, tp = confusion_matrix(_y_true, _y_pred).ravel()
            fpr = fp / float(tn + fp)
            fnr = fn / float(tp + fn)
            f1 = f1_score(_y_true, _y_pred, average='binary')
            #print("Other evaluation metrics we may need:")
            print(f"False Negative Rate (FNR) is {fnr * 100:.2f}%, False Positive Rate (FPR) is {fpr * 100:.2f}%, F1 score is {f1 * 100:.2f}%")



        if indicator_masking:
            flag_of_retaining = indicator_flag  # excluding the examples with ``not sure'' response
            y_pred = y_pred[flag_of_retaining]
            y_true = y_true[flag_of_retaining]
            ExS = (len(y_cent) - len(y_true))/len(y_cent)
            print('Deferred option : ')
            print('The threshold is {:.5}'.format(self.tau.item()))
            print(f"Percentage of excluded samples is {ExS * 100:.2f}%")
            measurement(y_true, y_pred)

        else:
            # instead filtering out examples, here resets the prediction as 1 (test model on perturbed examples)
            y_pred[~indicator_flag] = 1.
            print('Conservative option : ')
            print('The threshold is {:.5}'.format(self.tau.item()))
            measurement(y_true, y_pred)



    def inference(self, test_data_producer):
        y_cent, x_prob = [], []
        gt_labels = []
        self.eval()
        with torch.no_grad():
            for x, y in test_data_producer:
                x, y = to_device(x.double(), y.long(), self.device)
                logits_f, logits_g = self.forward(x)
                y_cent.append(torch.softmax(logits_f, dim=-1))
                x_prob.append(logits_g)
                gt_labels.append(y)

        gt_labels = torch.cat(gt_labels, dim=0)
        y_cent = torch.cat(y_cent, dim=0)
        x_prob = torch.cat(x_prob, dim=0)
        return y_cent, x_prob, gt_labels


    def inference_batch_wise(self, x):
        assert isinstance(x, torch.Tensor)
        self.eval()
        logits_f, logits_g = self.forward(x)
        return torch.softmax(logits_f, dim=-1).detach().cpu().numpy(), logits_g.detach().cpu().numpy()

    def get_tau_sample_wise(self, y_pred=None):
        return self.tau

    def indicator(self, x_prob, y_pred=None):
        """
        Return 'True' if a sample is original, and otherwise 'False' is returned.
        """
        if isinstance(x_prob, np.ndarray):
            x_prob = torch.tensor(x_prob, device=self.device)
            return (x_prob <= self.tau).cpu().numpy()
        elif isinstance(x_prob, torch.Tensor):
            return x_prob <= self.tau
        else:
            raise TypeError("Tensor or numpy.ndarray are expected.")

    def get_threshold(self, validation_data_producer, ratio=None):
        """
        get the threshold for adversary detection
        :@param validation_data_producer: Object, an iterator for producing validation dataset
        """
        self.eval()
        ratio = ratio if ratio is not None else self.ratio
        assert 0 <= ratio <= 1
        probabilities = []
        with torch.no_grad():
            for x_val, y_val in validation_data_producer:
                x_val, y_val = to_tensor(x_val.double(), y_val.long(), self.device)
                x_logits = self.forward_g(x_val)
                probabilities.append(x_logits)
            s, _ = torch.sort(torch.cat(probabilities, dim=0))
            i = int((s.shape[0] - 1) * ratio)
            assert i >= 0
            self.tau[0] = s[i]

    def reset_threshold(self):
        self.tau[0] = 0.


    def load(self, model_save_path):
        # load model
        self.load_state_dict(torch.load(model_save_path, map_location=torch.device(device)))

    # loading PAD
    def load2(self, model_save_path):
        ckpt = torch.load(model_save_path, map_location=torch.device(device))
        self.load_state_dict(ckpt['model'])

### Step 2 — The attack and evaluation functions

# Attack & Evaluations

In [ ]:
import torch
import torch.optim.lr_scheduler as lr_scheduler
from torch import Tensor, nn


# success_mask: success samples in inner loop
# success_mask_R: success rounded samples in inner loop
# success_samples: total success samples in inner loop
# success_samples_R: total success rounded samples in inner loop
# o_success_samples: total success samples
# o_success_samples_R: total success rounded samples

# Helper Function: Forward Pass
def forward_pass(inputs: Tensor, model: nn.Module, enable_detector: bool, oblivion: bool):
    """
    Perform a forward pass through the model, optionally enabling the detector.
    """
    if enable_detector:
        if model.name == 'KDE':
            logits, hidden_layer = model.forward_f(inputs)
            probabilities = model.forward_g(hidden_layer, y_pred=0)
        else:
            logits, probabilities = model.forward(inputs)
    else:
        logits, probabilities = model.forward(inputs), None
    return logits, probabilities


# Helper Function: Update Confidence
def update_confidence(confidence: Tensor, mask_success: Tensor, mask_failure: Tensor, bound: float) -> Tensor:
    """
    Adjust confidence values for successful and failed samples.
    """
    confidence[mask_success] = -bound
    confidence[mask_failure] = bound
    return confidence


# Helper Function: Update Bounds and Constants
def update_bounds_and_consts(
    outer_step: int,
    binary_search_steps: int,
    CONST: torch.Tensor,
    upper_bound: torch.Tensor,
    lower_bound: torch.Tensor,
    success_samples_R: torch.Tensor,
):
    """
    Update bounds and constants for binary search.
    """
    if (outer_step + 1) == binary_search_steps:
        return CONST, upper_bound, lower_bound

    # Update bounds
    upper_bound[success_samples_R] = torch.minimum(upper_bound[success_samples_R], CONST[success_samples_R])
    lower_bound[~success_samples_R] = torch.maximum(lower_bound[~success_samples_R], CONST[~success_samples_R])

    # Update CONST values
    CONST[success_samples_R] /= 100.0
    CONST[~success_samples_R] *= 100.0

    # Adjust bounds dynamically
    updated_upper_limit_mask = upper_bound < 1e8
    updated_lower_limit_mask = lower_bound > 0

    CONST[success_samples_R & updated_lower_limit_mask] = (
        (lower_bound[success_samples_R & updated_lower_limit_mask] +
         upper_bound[success_samples_R & updated_lower_limit_mask]) / 2.0
    )
    CONST[~success_samples_R & updated_upper_limit_mask] = (
        (lower_bound[~success_samples_R & updated_upper_limit_mask] +
         upper_bound[~success_samples_R & updated_upper_limit_mask]) / 2.0
    )

    return CONST, upper_bound, lower_bound



# Main Function: Sigma Zero Rounded Optimization
def sigma_zero(
    model: nn.Module,
    original_inputs: Tensor,
    feature_mask: Tensor,
    max_iterations: int = 1000,
    learning_rate: float = 0.5,
    sigma: float = 1e-4,
    threshold: float = 0.3,
    verbose: bool = False,
    grad_norm: float = torch.inf,
    detector_enabled: bool = False,
    oblivion: bool = False,
    binary_search_steps: int = 4,
    initial_const = 1.0,
    primary_confidence: float = 1e-4,
    secondary_confidence: float = 1e-4,
    device: str = 'cpu',
) -> Tensor:
    """
    Optimizes adversarial examples with L0 constraints using iterative methods.

    Parameters:
    - model (nn.Module): The model to evaluate adversarial examples.
    - original_inputs (Tensor): The original input samples.
    - feature_mask (Tensor): Mask indicating features that should remain fixed.
    - max_iterations (int): The maximum number of iterations for optimization.
    - learning_rate (float): The learning rate for the optimizer.
    - sigma (float): A small constant to control the L0 approximation.
    - threshold (float): A threshold for making zero those features with small perturbation.
    - verbose (bool): Whether to print detailed progress information.
    - grad_norm (float): The gradient normalization parameter.
    - detector_enabled (bool): Whether to use the detector model for evaluation.
    - oblivion (bool): Flag to determine if the oblivion mode should be applied.
    - binary_search_steps (int): The number of binary search steps for adjusting the confidence bounds.
    - initial_const (float): The initial value for the constant used in the loss function.
    - primary_confidence (float): confidence for the primary loss term.
    - secondary_confidence (float): confidence for the secondary loss term.
    - device (str): The device ('cpu' or 'cuda') where the tensors are located.

    Returns:
    - (inputs + best_delta) (Tensor): The optimized adversarial examples.
    """

    # Initialization
    model.eval()
    original_inputs = original_inputs.to(device)
    batch_size, num_features = original_inputs.shape

    # Create mask for non-fixed features
    modifiable_features_mask = torch.bitwise_or(
        feature_mask.expand(batch_size, -1), 1 - original_inputs.to(torch.uint8)
    ).bool()

    # Initialize tracking variables
    best_dist = torch.full((batch_size,), num_features, device=device, dtype=torch.float32)
    best_delta = torch.zeros_like(original_inputs, device=device)
    o_success_samples = torch.zeros(batch_size, dtype=torch.bool, device=device)

    # Initialize CONST and bounds
    CONST = torch.full((batch_size,), initial_const, dtype=torch.float32, device=device)
    lower_bound = torch.zeros(batch_size, dtype=torch.float32, device=device)
    upper_bound = torch.full((batch_size,), 1e8, dtype=torch.float32, device=device)

    # Outer loop for binary search
    outer_iters = binary_search_steps if (detector_enabled and not oblivion) else 1
    for outer_step in range(outer_iters):
        if verbose and detector_enabled and not oblivion:
            print(f"{'-'*50}")
            print(f"Outer step: {outer_step + 1}, CONST: {torch.unique(CONST)}")

        # Initialize perturbation delta
        delta = torch.zeros_like(original_inputs, requires_grad=True, device=device)
        delta.register_hook(lambda grad: grad * modifiable_features_mask)

        # Optimizer and scheduler
        optimizer = torch.optim.Adam([delta], lr=learning_rate)
        scheduler = lr_scheduler.CosineAnnealingLR(optimizer, T_max=max_iterations, eta_min=learning_rate / 100.0)

        # Initialize dynamic variables
        threshold_matrix = torch.full_like(original_inputs, threshold, device=device)
        success_samples = torch.zeros(batch_size, dtype=torch.bool, device=device)

        # Inner optimization loop
        for step in range(max_iterations):
            adv_inputs = original_inputs + delta
            logits, prob = forward_pass(adv_inputs, model, detector_enabled, oblivion)

            # Compute losses
            ben_score, mal_score = logits[:, 0], logits[:, 1]
            tau = model.tau[0].item() if detector_enabled and not oblivion else 0


            # Update success masks and distances
            with torch.no_grad():
                success_mask = (logits.argmax(dim=1) == 0) & ((prob <= tau) if detector_enabled and not oblivion else True)
                success_samples |= success_mask
                o_success_samples |= success_mask


                dist = (adv_inputs != original_inputs).float().sum(dim=1)
                improved_mask = success_mask & (dist < best_dist)
                best_dist = torch.where(improved_mask, dist, best_dist)
                best_delta = torch.where(improved_mask.unsqueeze(-1), delta.data.clone(), best_delta)

            primary_loss = torch.clamp(mal_score - ben_score + primary_confidence, min=0.0)
            if detector_enabled and not oblivion:
                primary_loss += CONST * torch.clamp(prob - tau + secondary_confidence, min=0.0)

            # Combined loss and optimization
            l0_approx = delta.square() / (delta.square() + sigma)
            l0_normalized = l0_approx.sum(dim=1) / num_features
            total_loss_mean = (primary_loss + l0_normalized).mean()


            if verbose and (step == 0 or (step + 1) % 2000 == 0):
                avg_best_dist = best_dist[o_success_samples].float().mean().item() if o_success_samples.any() else float('nan')
                print(
                    f"Iteration {step + 1:5} | "
                    f"LR = {scheduler.get_last_lr()[0]:.2f} | "
                    f"Current Success: {success_mask.sum().item():4}/{success_samples.sum().item():4} | "
                    f"Loop Success = {success_samples.sum().item():4}/{batch_size} | "
                    f"Total Success: {o_success_samples.sum().item():4}/{batch_size} | "
                    f"Best Dist (avg) = {avg_best_dist:3.2f} | "
                    f"Loss = {total_loss_mean.item():5.2f} | "
                )

            optimizer.zero_grad()
            total_loss_mean.backward()
            delta.grad /= delta.grad.norm(p=grad_norm, dim=1, keepdim=True).clamp_min(1e-12)
            optimizer.step()
            scheduler.step()

            # Enforce bounds and threshold
            with torch.no_grad():
                delta.data.add_(original_inputs).clamp_(0, 1).sub_(original_inputs)

                threshold_matrix[~success_mask] -= 0.01 * scheduler.get_last_lr()[0]
                threshold_matrix[success_mask] += 0.01 * scheduler.get_last_lr()[0]
                threshold_matrix.clamp_(0, 1)

                delta.data[(delta.data.abs() < threshold_matrix)] = 0

        # Update bounds and constants
        CONST, upper_bound, lower_bound = update_bounds_and_consts(
            outer_step, binary_search_steps, CONST, upper_bound, lower_bound, success_samples
        )

    return (original_inputs + best_delta)

In [ ]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from typing import Any, Tuple

def attack_mal_batches(
    test_loader: DataLoader,
    model: nn.Module,
    attack: Any,
    feature_mask: torch.Tensor,
    device: torch.device,
    detector_enabled: bool = True,
    oblivion: bool = False,
    indicator_masking: bool = False,
    **kwargs
) -> Tuple[torch.Tensor, torch.Tensor]:
    """
    Applies adversarial attacks on malicious samples within batches and returns
    the original malicious samples and the corresponding adversarial examples.

    Args:
        test_loader (DataLoader): DataLoader for test data.
        model (nn.Module): Model to be attacked.
        attack (Any): Adversarial attack function (e.g., sigma_zero_rounded).
        feature_mask (torch.Tensor): Masking tensor indicating removable features.
        device (torch.device): Device for computation.
        detector_enabled (bool, optional): Enables/disables detector during attack.
        oblivion (bool, optional): Flag for attack configuration.
        indicator_masking (bool, optional): Enables indicator masking for suspecious samples (prob>tau).
        **kwargs: Additional parameters for the attack function (e.g., steps, lr).

    Returns:
        Tuple[torch.Tensor, torch.Tensor]: Original malicious samples and adversarial examples.
    """
    model.eval()
    perturbed_batches = []
    perturbed_indicator_batches = []
    mal_x_batches = []

    # Iterate through each batch in the test_loader
    for x_test, y_test in test_loader:
        torch.cuda.empty_cache()

        # Move batch to device
        x_test, y_test = x_test.to(device), y_test.to(device)

        # Select malicious samples (label == 1)
        mal_x_batch = x_test[y_test.squeeze() == 1]

        if mal_x_batch.size(0) > 0:  # Process only if there are malicious samples
            # Generate adversarial examples for malicious samples
            perturbed_batch = attack(
                model=model,
                original_inputs=mal_x_batch,
                feature_mask=feature_mask,
                device=device,
                detector_enabled=detector_enabled,
                oblivion=oblivion,
                **kwargs
            )
            perturbed_batches.append(perturbed_batch)
            mal_x_batches.append(mal_x_batch)

            # Apply indicator masking if enabled and required
            if detector_enabled and not oblivion and indicator_masking:
                print(f"{'-'*50}")
                print('Oblivion Attack ')
                perturbed_indicator_batch = attack(
                    model=model,
                    inputs=mal_x_batch,
                    feature_mask=feature_mask,
                    device=device,
                    detector_enabled=detector_enabled,
                    oblivion=True,
                    **kwargs
                )
                perturbed_indicator_batches.append(perturbed_indicator_batch)

    # Combine all malicious samples and their perturbed counterparts
    combined_mals = torch.cat(mal_x_batches, dim=0)
    combined_advs = torch.cat(perturbed_batches, dim=0)

    # Initialize final adversarial examples based on attack conditions
    if detector_enabled and not oblivion and indicator_masking:
        combined_indicator_advs = torch.cat(perturbed_indicator_batches, dim=0)
        final_advs = combined_advs.clone()

        # Get logits and probability
        if model.name == 'KDE':
            logits, x_hidden = model.forward_f(final_advs)
            prob = model.forward_g(x_hidden, y_pred=0)
        else:
            logits, prob = model(final_advs)

        tau = model.tau[0].item()
        success_mask = (logits.argmax(dim=1) == 0) & (prob <= tau)

        # Update unsuccessful adversarial examples using the indicator batch
        final_advs = torch.where(success_mask.unsqueeze(-1), final_advs, combined_indicator_advs)
    else:
        final_advs = combined_advs

    return combined_mals, final_advs

In [ ]:
import torch
import torch.optim.lr_scheduler as lr_scheduler
from torch import Tensor, nn


# success_mask: success samples in inner loop
# success_mask_R: success rounded samples in inner loop
# success_samples: total success samples in inner loop
# success_samples_R: total success rounded samples in inner loop
# o_success_samples: total success samples
# o_success_samples_R: total success rounded samples

# Helper Function: Forward Pass
def forward_pass(inputs: Tensor, model: nn.Module, enable_detector: bool, oblivion: bool):
    """
    Perform a forward pass through the model, optionally enabling the detector.
    """
    if enable_detector:
        if model.name == 'KDE':
            logits, hidden_layer = model.forward_f(inputs)
            if not oblivion:
                probabilities = model.forward_g(hidden_layer, y_pred=0)
            else:
                probabilities = None
        else:
            logits, probabilities = model.forward(inputs)
    else:
        logits, probabilities = model.forward(inputs), None
    return logits, probabilities


# Helper Function: Update Confidence
def update_confidence(confidence: Tensor, mask_success: Tensor, mask_failure: Tensor, bound: float) -> Tensor:
    """
    Adjust confidence values for successful and failed samples.
    """
    confidence[mask_success] = -bound
    confidence[mask_failure] = bound
    return confidence


# Helper Function: Update Bounds and Constants
def update_bounds_and_consts(
    outer_step: int,
    binary_search_steps: int,
    CONST: torch.Tensor,
    upper_bound: torch.Tensor,
    lower_bound: torch.Tensor,
    success_samples_R: torch.Tensor,
):
    """
    Update bounds and constants for binary search.
    """
    if (outer_step + 1) == binary_search_steps:
        return CONST, upper_bound, lower_bound

    # Update bounds
    upper_bound[success_samples_R] = torch.minimum(upper_bound[success_samples_R], CONST[success_samples_R])
    lower_bound[~success_samples_R] = torch.maximum(lower_bound[~success_samples_R], CONST[~success_samples_R])

    # Update CONST values
    CONST[success_samples_R] /= 100.0
    CONST[~success_samples_R] *= 100.0

    # Adjust bounds dynamically
    updated_upper_limit_mask = upper_bound < 1e8
    updated_lower_limit_mask = lower_bound > 0

    CONST[success_samples_R & updated_lower_limit_mask] = (
        (lower_bound[success_samples_R & updated_lower_limit_mask] +
         upper_bound[success_samples_R & updated_lower_limit_mask]) / 2.0
    )
    CONST[~success_samples_R & updated_upper_limit_mask] = (
        (lower_bound[~success_samples_R & updated_upper_limit_mask] +
         upper_bound[~success_samples_R & updated_upper_limit_mask]) / 2.0
    )

    return CONST, upper_bound, lower_bound



# Main Function: Sigma Zero Rounded Optimization
def sigma_zero_rounded_alpha(
    model: nn.Module,
    original_inputs: Tensor,
    feature_mask: Tensor,
    max_iterations: int = 1000,
    learning_rate: float = 0.5,
    sigma: float = 1e-4,
    threshold: float = 0.3,
    verbose: bool = False,
    grad_norm: float = torch.inf,
    detector_enabled: bool = False,
    oblivion: bool = False,
    binary_search_steps: int = 5,
    initial_const = 1.0,
    initial_primary_confidence: float = 1e-4,
    initial_secondary_confidence: float = 1e-4,
    device: str = 'cpu',
    confidence_bound_primary: float = 1.,
    confidence_bound_secondary: float = 1.,
    confidence_update_interval: int = 50,
) -> Tensor:
    """
    Optimizes adversarial examples with L0 constraints using iterative methods.

    Parameters:
    - model (nn.Module): The model to evaluate adversarial examples.
    - original_inputs (Tensor): The original input samples.
    - feature_mask (Tensor): Mask indicating features that should remain fixed.
    - max_iterations (int): The maximum number of iterations for optimization.
    - learning_rate (float): The learning rate for the optimizer.
    - sigma (float): A small constant to control the L0 approximation.
    - threshold (float): A threshold for making zero those features with small perturbation.
    - verbose (bool): Whether to print detailed progress information.
    - grad_norm (float): The gradient normalization parameter.
    - detector_enabled (bool): Whether to use the detector model for evaluation.
    - oblivion (bool): Flag to determine if the oblivion mode should be applied.
    - binary_search_steps (int): The number of binary search steps for adjusting the confidence bounds.
    - initial_const (float): The initial value for the constant used in the loss function.
    - initial_primary_confidence (float): Initial confidence for the primary loss term.
    - initial_secondary_confidence (float): Initial confidence for the secondary loss term.
    - device (str): The device ('cpu' or 'cuda') where the tensors are located.
    - primary_confidence_bound (float): The bound for the primary confidence term.
    - secondary_confidence_bound (float): The bound for the secondary confidence term.
    - confidence_update_interval (int): The interval at which to update the confidence values.

    Returns:
    - final_adv (Tensor): The optimized adversarial examples.
    """

    # Initialization
    model.eval()
    original_inputs = original_inputs.to(device)
    batch_size, num_features = original_inputs.shape

    # Create mask for non-fixed features
    modifiable_features_mask = torch.bitwise_or(
        feature_mask.expand(batch_size, -1), 1 - original_inputs.to(torch.uint8)
    ).bool()

    # Initialize tracking variables
    best_dist = torch.full((batch_size,), num_features, device=device, dtype=torch.float32)
    best_delta = torch.zeros_like(original_inputs, device=device)
    o_success_samples = torch.zeros(batch_size, dtype=torch.bool, device=device)
    o_success_samples_R = torch.zeros(batch_size, dtype=torch.bool, device=device)

    # Initialize CONST and bounds
    CONST = torch.full((batch_size,), initial_const, dtype=torch.float32, device=device)
    lower_bound = torch.zeros(batch_size, dtype=torch.float32, device=device)
    upper_bound = torch.full((batch_size,), 1e8, dtype=torch.float32, device=device)

    # Outer loop for binary search
    outer_iters = binary_search_steps if (detector_enabled and not oblivion) else 1
    for outer_step in range(outer_iters):
        if verbose and detector_enabled and not oblivion:
            print(f"{'-'*50}")
            print(f"Outer step: {outer_step + 1}, CONST: {torch.unique(CONST)}")

        # Initialize perturbation delta
        delta = torch.zeros_like(original_inputs, requires_grad=True, device=device)
        delta.register_hook(lambda grad: grad * modifiable_features_mask)

        # Optimizer and scheduler
        optimizer = torch.optim.Adam([delta], lr=learning_rate)
        scheduler = lr_scheduler.CosineAnnealingLR(optimizer, T_max=max_iterations, eta_min=learning_rate / 100.0)

        # Initialize dynamic variables
        threshold_matrix = torch.full_like(original_inputs, threshold, device=device)
        primary_confidence = torch.full((batch_size,), initial_primary_confidence, device=device)
        secondary_confidence = torch.full((batch_size,), initial_secondary_confidence, device=device)
        success_samples = torch.zeros(batch_size, dtype=torch.bool, device=device)
        success_samples_R = torch.zeros(batch_size, dtype=torch.bool, device=device)
        stagnation_counter = torch.zeros(batch_size, dtype=torch.int32, device=device)
        best_loss_inloop = torch.full((batch_size,), float('inf'), device=device, dtype=torch.float32)

        o_stagnated_mask = torch.zeros(batch_size, dtype=torch.bool, device=device)

        # Inner optimization loop
        for step in range(max_iterations):
            adv_inputs = original_inputs + delta
            logits, prob = forward_pass(adv_inputs, model, detector_enabled, oblivion)

            # Compute losses
            ben_score, mal_score = logits[:, 0], logits[:, 1]
            tau = model.tau[0].item() if detector_enabled and not oblivion else 0


            # Update success masks and distances
            with torch.no_grad():
                success_mask = (logits.argmax(dim=1) == 0) & ((prob <= tau) if detector_enabled and not oblivion else True)
                success_samples |= success_mask
                o_success_samples |= success_mask

                if step == 0 or (step + 1) % 10 == 0:
                    adv_inputs_rounded = prioritized_binary_rounding(original_inputs, adv_inputs, model, feature_mask, detector_enabled, oblivion)
                    rounded_logits, rounded_prob = forward_pass(adv_inputs_rounded, model, detector_enabled, oblivion)

                    success_mask_R = (rounded_logits.argmax(dim=1) == 0) & ((rounded_prob <= tau) if detector_enabled and not oblivion else True)
                    success_samples_R |= success_mask_R
                    o_success_samples_R |= success_mask_R

                    dist = (adv_inputs_rounded != original_inputs).float().sum(dim=1)
                    improved_mask = success_mask_R & (dist < best_dist)
                    best_dist = torch.where(improved_mask, dist, best_dist)
                    best_delta = torch.where(improved_mask.unsqueeze(-1), delta.data.clone(), best_delta)


                if (step + 1) % confidence_update_interval == 0:
                  if o_stagnated_mask.any():
                      primary_confidence = update_confidence(primary_confidence, o_stagnated_mask & success_mask, o_stagnated_mask & ~success_mask, confidence_bound_primary)
                      if detector_enabled and not oblivion:
                          secondary_confidence = update_confidence(secondary_confidence, o_stagnated_mask & success_mask, o_stagnated_mask & ~success_mask, confidence_bound_secondary)

            primary_loss = torch.clamp(mal_score - ben_score + primary_confidence, min=0.0)
            if detector_enabled and not oblivion:
                primary_loss += CONST * torch.clamp(prob - tau + secondary_confidence, min=0.0)

            # Combined loss and optimization
            l0_approx = delta.square() / (delta.square() + sigma)
            l0_normalized = l0_approx.sum(dim=1) / num_features
            total_loss = (primary_loss + l0_normalized)
            total_loss_mean = total_loss.mean()

            with torch.no_grad():
                # Track the best loss and stagnation counter
                improved_mask_inloop = success_mask & (total_loss < best_loss_inloop)
                best_loss_inloop = torch.where(improved_mask_inloop, total_loss, best_loss_inloop)

                stagnation_counter = torch.where(improved_mask_inloop, 0, stagnation_counter + 1)
                o_stagnated_mask |= ((stagnation_counter >= 1000) & success_samples)




            if verbose and (step == 0 or (step + 1) % 1000 == 0):
                avg_best_dist = best_dist[o_success_samples_R].float().mean().item() if o_success_samples_R.any() else float('nan')
                print(
                    f"Iteration {step + 1:5} | "
                    f"LR = {scheduler.get_last_lr()[0]:.2f} | "
                    f"Current NotRound/Round = {success_mask.sum().item():4}/{success_mask_R.sum().item():4} | "
                    f"Loop NotRounded/Rounded = {success_samples.sum().item():4}/{success_samples_R.sum().item():4} | "
                    f"Total NotRound/Round = {o_success_samples.sum().item():4}/{o_success_samples_R.sum().item():4} | "
                    f"Best Dist (avg) = {avg_best_dist:3.2f} | "
                    f"Loss = {total_loss_mean.item():5.2f} | "
                )

            optimizer.zero_grad()
            total_loss_mean.backward()
            delta.grad /= delta.grad.norm(p=grad_norm, dim=1, keepdim=True).clamp_min(1e-12)
            optimizer.step()
            scheduler.step()

            # Enforce bounds and threshold
            with torch.no_grad():
                delta.data.add_(original_inputs).clamp_(0, 1).sub_(original_inputs)

                threshold_matrix[~success_mask] -= 0.01 * scheduler.get_last_lr()[0]
                threshold_matrix[success_mask] += 0.01 * scheduler.get_last_lr()[0]
                threshold_matrix.clamp_(0, 1)

                delta.data[(delta.data.abs() < threshold_matrix)] = 0

        # Update bounds and constants
        CONST, upper_bound, lower_bound = update_bounds_and_consts(
            outer_step, binary_search_steps, CONST, upper_bound, lower_bound, success_samples_R
        )

    # Final outputs
    final_adv =  prioritized_binary_rounding(original_inputs, (original_inputs + best_delta).detach(), model, feature_mask, detector_enabled, oblivion)
    final_adv =  prioritized_binary_rounding(original_inputs, final_adv.detach(), model, feature_mask, detector_enabled, oblivion)

    return final_adv

In [ ]:
def prioritized_binary_rounding(original_inputs, adversarial_inputs, model, feature_mask, detector_enabled=False, oblivion=False):
    """
    Apply rounding adjustments to adversarial examples by prioritizing changes based on model's predictions.

    Parameters:
    - original_inputs (Tensor): The original input tensor.
    - adversarial_inputs (Tensor): The adversarially perturbed input tensor.
    - model (nn.Module): The model used for evaluation of the input samples.
    - feature_mask (Tensor): A tensor indicating which features should remain unchanged.
    - detector_enabled (bool): Flag indicating whether a detector is enabled (optional).
    - oblivion (bool): Flag indicating whether the attacker is unaware of a detector (optional).

    Returns:
    - final_inputs (Tensor): The adjusted adversarial examples after rounding.
    """

    # Ensure all tensors are on the same device
    device = original_inputs.device
    original_inputs, adversarial_inputs, feature_mask = original_inputs.to(device), adversarial_inputs.to(device), feature_mask.to(device)

    # Create mask for features that can be modified (1 means modifiable, 0 means fixed)
    modifiable_mask = torch.bitwise_or(feature_mask.expand(original_inputs.shape[0], -1), 1 - original_inputs.to(torch.uint8)).bool()

    # Set the model to evaluation mode
    model.eval()

    # Compute model predictions and prepare for change prioritization
    with torch.no_grad():

        # Model inference with or without detector
        if detector_enabled :
            if model.name == 'KDE':
                logits, hidden_layer = model.forward_f(original_inputs)
                if not oblivion:
                    probabilities = model.forward_g(hidden_layer, y_pred=0)
                else :
                    probabilities = None
            else:
                logits, probabilities = model.forward(original_inputs)
        else:
            logits, probabilities = model.forward(original_inputs), None


        # If detector is enabled and we don't ignore it, create a mask for inactive samples
        if detector_enabled and not oblivion:
            threshold = model.tau[0].item()
            inactive_mask = (logits.argmax(dim=1) == 0) & (probabilities <= threshold)
        else:
            inactive_mask = (logits.argmax(dim=1) == 0)

        # Active samples are those that are not inactive
        active_samples_mask = ~inactive_mask

        # Calculate absolute changes between adversarial and original inputs
        change_magnitude = (adversarial_inputs - original_inputs).abs()

        # Create a mask for significant changes (greater than a threshold)
        significant_change_mask = change_magnitude > 0.1
        sorted_change_indices = change_magnitude.argsort(dim=1, descending=True, stable=True)

        # Determine the changes to be applied based on adversarial inputs
        proposed_changes = torch.where(adversarial_inputs > original_inputs, 1,
                                       torch.where(adversarial_inputs < original_inputs, 0, original_inputs)).float().to(device)

    # Clone original inputs to apply final adjustments
    final_inputs = original_inputs.clone()

    # Set a reasonable upper bound on the number of significant changes per sample
    max_changes_per_sample = min(significant_change_mask.sum(dim=1).max().item(), 300)

    # Apply changes iteratively, ensuring that only non-fixed features are modified
    for i in range(max_changes_per_sample):
        if not active_samples_mask.any():
            break

        active_sample_indices = torch.where(active_samples_mask)[0]

        # Update final input tensor based on the sorted indices and modifiable mask
        indices_to_change = sorted_change_indices[active_sample_indices, i]
        final_inputs[active_sample_indices, indices_to_change] = torch.where(
            modifiable_mask[active_sample_indices, indices_to_change],
            proposed_changes[active_sample_indices, indices_to_change],
            final_inputs[active_sample_indices, indices_to_change]
        )

        # Update the active mask by re-evaluating the model predictions after applying changes
        with torch.no_grad():

            if detector_enabled :
                if model.name == 'KDE':
                    logits, hidden_layer = model.forward_f(final_inputs[active_sample_indices])
                    if not oblivion:
                        probabilities = model.forward_g(hidden_layer, y_pred=0)
                    else :
                        probabilities = None
                else:
                    logits, probabilities = model.forward(final_inputs[active_sample_indices])
            else:
                logits, probabilities = model.forward(final_inputs[active_sample_indices]), None


            # Recompute the inactive mask if detector is enabled and update the active mask accordingly
            if detector_enabled and not oblivion:
                inactive_mask = (logits.argmax(dim=1) == 0) & (probabilities <= threshold)
            else:
                inactive_mask = (logits.argmax(dim=1) == 0)

            # Update the active sample mask to reflect which samples remain active
            active_samples_mask[active_sample_indices] = (~inactive_mask)

    return final_inputs

In [ ]:
def ensure_fixed_elements_unchanged(mal_x_batch, adv, feature_mask):
    """
    Ensure that fixed elements are not changed and calculate differences between adversarial examples and the original batch.

    Parameters:
    - mal_x_batch: Original malware input batch
    - adv: Adversarial examples generated from mal_x_batch
    - feature_mask: Array indicating the fixed features

    Returns:
    - A dictionary containing differences for fixed and non-fixed features
    """

    # Ensure `feature_mask` is expanded to match the batch size
    expanded_feature_mask = feature_mask.expand(mal_x_batch.shape[0], -1)

    # Create a mask where non-fixed features are set to 1
    non_fixed_features_mask = torch.bitwise_or(expanded_feature_mask, 1 - mal_x_batch.to(torch.uint8))

    # Calculate indices where the mask is 0 (fixed) or 1 (non-fixed)
    mask_fixed = non_fixed_features_mask == 0
    mask_non_fixed = non_fixed_features_mask == 1

    # Calculate the differences for fixed and non-fixed features
    diff_adv_fixed = (torch.abs(adv - mal_x_batch) * mask_fixed).sum().item()
    diff_adv_non_fixed = (torch.abs(adv - mal_x_batch) * mask_non_fixed).sum().item()

    # Print the differences
    print(f"Differences when mask is 0 (fixed features):")
    print(f"  Adv vs. Original: {diff_adv_fixed}")

    print(f"Differences when mask is 1 (non-fixed features):")
    print(f"  Adv vs. Original: {diff_adv_non_fixed}")

In [ ]:
import torch
from torch import nn
from typing import List

def calculate_distance(adv_samples: torch.Tensor, original_samples: torch.Tensor, norm: str) -> torch.Tensor:
    """
    Calculates the distance between adversarial and original samples based on the specified norm.

    Args:
        adv_samples (torch.Tensor): The adversarial samples.
        original_samples (torch.Tensor): The original samples.
        norm (str): The norm type ('L0', 'L1', 'L2').

    Returns:
        torch.Tensor: Calculated distances between each adversarial and original sample.
    """
    if norm == 'L0':
        return (adv_samples != original_samples).float().sum(dim=1)
    elif norm == 'L1':
        return torch.abs(adv_samples - original_samples).sum(dim=1)
    elif norm == 'L2':
        return torch.norm(adv_samples - original_samples, p=2, dim=1)
    else:
        raise ValueError("Unsupported norm. Use 'L0', 'L1', or 'L2'.")

def median_of_top_n_distances(adv_rounded: torch.Tensor, mal_x_batch: torch.Tensor, y_pred: torch.Tensor, n: int, norm: str = 'L0') -> float:
    """
    Calculate the median of the top n smallest distances between adv_rounded and mal_x_batch
    for samples where y_pred == 0, based on the specified norm.

    Args:
        adv_rounded (torch.Tensor): The rounded adversarial examples tensor.
        mal_x_batch (torch.Tensor): The original malicious examples tensor.
        y_pred (torch.Tensor): The prediction tensor.
        n (int): The number of top distances to consider.
        norm (str): The norm to use ('L0', 'L1', 'L2').

    Returns:
        float: The median of the top n smallest distances.
    """
    distances = calculate_distance(adv_rounded[y_pred == 0], mal_x_batch[y_pred == 0], norm)
    top_n_distances = torch.topk(distances, n, largest=False).values
    return top_n_distances.median().item()

def evaluate_adversarial_examples(
    mal_x: torch.Tensor,
    adv: torch.Tensor,
    model: nn.Module,
    feature_mask: torch.Tensor,
    detector_enabled: bool = False,
    oblivion: bool = False,
    indicator_masking: bool = False,
    rounded: bool = False,
    batch_size: int = 1024,
    norm: str = 'L0'
) -> torch.Tensor:
    """
    Evaluates adversarial examples against a model, calculating various distance metrics
    and success rates for different norms, optionally in batches.

    Args:
        mal_x (torch.Tensor): Original malware input.
        adv (torch.Tensor): Adversarial examples generated from mal_x.
        model (nn.Module): The trained model to be evaluated.
        feature_mask (torch.Tensor): Array used for final rounding.
        detector_enabled (bool, optional): Enables detector processing during evaluation.
        oblivion (bool, optional): Alters attack configuration if true.
        indicator_masking (bool, optional): Enables indicator masking for suspect samples.
        rounded (bool, optional): Applies final rounding on adversarial examples.
        batch_size (int, optional): Maximum samples per batch for processing.
        norm (str): Norm used for distance calculations ('L0', 'L1', 'L2').

    Returns:
        torch.Tensor: The final adversarial examples used in evaluation.
    """
    model.eval()
    torch.cuda.empty_cache()

    total_samples = mal_x.shape[0]
    num_successful_attacks = 0
    total_distances = 0
    y_pred_list = []
    mal_x_list = []
    final_adv_list = []
    all_distances: List[float] = []

    for i in range(0, total_samples, batch_size):
        mal_x_batch = mal_x[i:i + batch_size]
        adv_batch = adv[i:i + batch_size]

        # Apply rounding if specified
        final_adv_batch = (
            prioritized_binary_rounding(mal_x_batch, adv_batch, model, feature_mask, detector_enabled, oblivion)
            if rounded else adv_batch.clone()
        )

        # Perform forward pass and determine successful attacks
        with torch.no_grad():
            if detector_enabled:
                if model.name == 'KDE':
                    logits, x_hidden = model.forward_f(final_adv_batch)
                    prob = model.forward_g(x_hidden, y_pred=0)
                else:
                    logits, prob = model(final_adv_batch)

                # Identify successful attacks
                successful_attacks_mask = (logits.argmax(dim=1) == 0) & (prob <= model.tau[0].item())

                # Exclude samples for indicator masking
                if indicator_masking:
                    exclusion_mask = (logits.argmax(dim=1) == 0) & (prob > model.tau[0].item())
                    successful_attacks_mask = successful_attacks_mask[~exclusion_mask]
                    mal_x_batch = mal_x_batch[~exclusion_mask]
                    final_adv_batch = final_adv_batch[~exclusion_mask]

            else:
                logits = model(final_adv_batch)
                successful_attacks_mask = (logits.argmax(dim=1) == 0)

        final_adv_list.append(final_adv_batch)
        y_pred_batch = (~successful_attacks_mask).long()
        y_pred_list.append(y_pred_batch)
        mal_x_list.append(mal_x_batch)

        # Calculate distances for successful attacks
        num_successful_attacks += successful_attacks_mask.sum().item()
        if successful_attacks_mask.any():
            distances = calculate_distance(final_adv_batch[successful_attacks_mask], mal_x_batch[successful_attacks_mask], norm)
            total_distances += distances.sum().item()
            all_distances.extend(distances.cpu().tolist())

    # Aggregate predictions and adversarial examples
    y_pred = torch.cat(y_pred_list)
    final_adv = torch.cat(final_adv_list)
    final_mal_x = torch.cat(mal_x_list)

    # Calculate metrics
    num_considered_malwares = len(y_pred)
    ASR = (y_pred == 0).float().mean().item() * 100
    mean_distance = total_distances / max(1, num_successful_attacks)
    median_distance = torch.median(torch.tensor(all_distances)).item() if all_distances else 0

    ExS = (total_samples - num_considered_malwares) / total_samples


    # Print results
    print(f"Evaluation on Adversarial Examples (Norm: {norm}, Rounded: {rounded})")
    print("-----------------------------------------------")
    print(f"Total malwares: {total_samples} and Considered malwares : {num_considered_malwares}")
    print(f"Percentage of excluded samples is {ExS * 100:.2f}%")
    print(f"Successful attacks: {num_successful_attacks}/{num_considered_malwares}")
    print(f"Attack success rate:: {ASR:.2f}%")
    print(f"Mean distance (successful): {mean_distance:.2f}")
    print(f"Median distance (successful): {median_distance:.2f}")

    # Update the loop for top-N distance metrics
    top_n_values = [1000, 2000, 3000]
    for top_n in top_n_values:
        if num_successful_attacks >= top_n:
            median_top_n_distance = median_of_top_n_distances(final_adv, final_mal_x, y_pred, top_n, norm=norm)
            print(f"Median distance (TOP {top_n} successful): {median_top_n_distance:.2f}")


    return final_adv, final_mal_x

In [ ]:
import torch
import seaborn as sns
import matplotlib.pyplot as plt
from tqdm import tqdm

def calculate_distances(adv_batch, mal_x_batch, mask, norm):
    """
    Calculates the distance between adversarial and original samples based on the specified norm.

    Parameters:
    - adv_batch (torch.Tensor): Adversarial samples batch.
    - mal_x_batch (torch.Tensor): Original samples batch.
    - mask (torch.Tensor): Boolean mask for successful attacks.
    - norm (str): The norm type ('L0', 'L1', 'L2').

    Returns:
    - torch.Tensor: The calculated distances.
    """
    adv_samples = adv_batch[mask]
    original_samples = mal_x_batch[mask]

    if norm == 'L0':
        return (adv_samples != original_samples).float().sum(dim=1)
    elif norm == 'L1':
        return torch.abs(adv_samples - original_samples).sum(dim=1)
    elif norm == 'L2':
        return torch.norm(adv_samples - original_samples, p=2, dim=1)
    else:
        raise ValueError("Unsupported norm. Use 'L0', 'L1', or 'L2'.")


def plot_asr_vs_distance(adv_rounded_list, labels, model, mal_x_list, detector_enabled=False, device='cpu', batch_size=512, norm='L0', interval=5):
    sns.set(style='whitegrid')
    colors = sns.color_palette('husl', len(adv_rounded_list))
    line_styles = ['-', '--', '-.', ':']
    markers = ['o', 's', '^', 'D', 'v']

    # Disable LaTeX rendering to avoid errors
    plt.rcParams['text.usetex'] = False

    plt.figure(figsize=(18, 12))
    model.eval()
    x_min, x_max = 0, 0

    for idx, (adv_rounded, mal_x, color, label) in tqdm(enumerate(zip(adv_rounded_list, mal_x_list, colors, labels)),
                                                  total=len(adv_rounded_list), desc="Processing Tensors"):
        adv_rounded = adv_rounded.to(device)
        num_samples = adv_rounded.size(0)
        all_distances = []

        for start in range(0, num_samples, batch_size):
            end = min(start + batch_size, num_samples)
            adv_batch = adv_rounded[start:end]
            mal_x_batch = mal_x[start:end]

            with torch.no_grad():
                if detector_enabled:
                    if model.name == 'kde':
                        logits, x_hidden = model.forward_f(adv_batch)
                        prob = model.forward_g(x_hidden, y_pred=0)
                    else:
                        logits, prob = model.forward(adv_batch)
                    successful_attacks_mask = (logits.argmax(dim=1) == 0) & (prob <= model.tau[0].item())
                else:
                    logits = model.forward(adv_batch)
                    successful_attacks_mask = (logits.argmax(dim=1) == 0)

            if successful_attacks_mask.any():
                distances = calculate_distances(adv_batch, mal_x_batch, successful_attacks_mask, norm)
                all_distances.append(distances)

        if all_distances:
            total_distances = torch.cat(all_distances).cpu()
            x_max = max(x_max, int(total_distances.max().item()))
            x_values = np.arange(x_max)


            # Calculate ASR for each distance threshold
            asr_values = [(total_distances <= threshold).float().sum().item()/ num_samples * 100 for threshold in x_values]

            sns.lineplot(x=x_values, y=asr_values, marker=markers[idx % len(markers)], color=color,
                         linestyle=line_styles[idx % len(line_styles)], linewidth=2, markersize=5, alpha=0.8, label=label)

    attack_type = 'adaptive attack' if detector_enabled else 'attack'

    plt.xlabel(f'Distance Budget ({norm} norm)', fontsize=14, labelpad=10)
    plt.ylabel('Attack Success Rate (ASR) (%)', fontsize=14, labelpad=10)
    plt.title(f'ASR (%) vs. Distance under {attack_type}', fontsize=16, pad=10)
    plt.legend(title='Adversarial Sets', fontsize=10, title_fontsize=12)

    # Set x-axis ticks at intervals
    plt.xticks(ticks=range(x_min, x_max+1, interval))

    # Set y-axis ticks for percentage values
    plt.yticks(ticks=range(0, 110, 10))

    plt.grid(True, which='major', linestyle='--', linewidth=0.5)
    plt.minorticks_on()
    plt.grid(True, which='minor', linestyle=':', linewidth=0.5)
    sns.despine()
    plt.tight_layout()
    plt.show()

In [ ]:
import torch
import seaborn as sns
import matplotlib.pyplot as plt
from tqdm import tqdm

def calculate_distances(adv_batch, mal_x_batch, mask, norm):
    """
    Calculates the distance between adversarial and original samples based on the specified norm.

    Parameters:
    - adv_batch (torch.Tensor): Adversarial samples batch.
    - mal_x_batch (torch.Tensor): Original samples batch.
    - mask (torch.Tensor): Boolean mask for successful attacks.
    - norm (str): The norm type ('L0', 'L1', 'L2').

    Returns:
    - torch.Tensor: The calculated distances.
    """
    adv_samples = adv_batch[mask]
    original_samples = mal_x_batch[mask]

    if norm == 'L0':
        return (adv_samples != original_samples).float().sum(dim=1)
    elif norm == 'L1':
        return torch.abs(adv_samples - original_samples).sum(dim=1)
    elif norm == 'L2':
        return torch.norm(adv_samples - original_samples, p=2, dim=1)
    else:
        raise ValueError("Unsupported norm. Use 'L0', 'L1', or 'L2'.")


def plot_asr_vs_distance(adv_rounded_list, labels, models, mal_x_list, detector_enabled=False, device='cpu', batch_size=512, norm='L0', interval=5):
    sns.set(style='whitegrid')
    colors = sns.color_palette('husl', len(adv_rounded_list))
    line_styles = ['-', '--', '-.', ':']
    markers = ['o', 's', '^', 'D', 'v']

    # Disable LaTeX rendering to avoid errors
    plt.rcParams['text.usetex'] = False

    plt.figure(figsize=(9, 6))
    x_min, x_max = 0, 0

    for idx, (model, adv_rounded, mal_x, color, label) in tqdm(enumerate(zip(models, adv_rounded_list, mal_x_list, colors, labels)),
                                                  total=len(adv_rounded_list), desc="Processing Tensors"):
        model.eval()
        adv_rounded = adv_rounded.to(device)
        num_samples = adv_rounded.size(0)
        all_distances = []

        for start in range(0, num_samples, batch_size):
            end = min(start + batch_size, num_samples)
            adv_batch = adv_rounded[start:end]
            mal_x_batch = mal_x[start:end]

            with torch.no_grad():
                if detector_enabled:
                    if model.name == 'kde':
                        logits, x_hidden = model.forward_f(adv_batch)
                        prob = model.forward_g(x_hidden, y_pred=0)
                    else:
                        logits, prob = model.forward(adv_batch)
                    successful_attacks_mask = (logits.argmax(dim=1) == 0) & (prob <= model.tau[0].item())
                else:
                    logits = model.forward(adv_batch)
                    successful_attacks_mask = (logits.argmax(dim=1) == 0)

            if successful_attacks_mask.any():
                distances = calculate_distances(adv_batch, mal_x_batch, successful_attacks_mask, norm)
                all_distances.append(distances)

        if all_distances:
            total_distances = torch.cat(all_distances).cpu()
            x_max = max(x_max, int(total_distances.max().item()))
            x_values = np.arange(x_max)


            # Calculate ASR for each distance threshold
            asr_values = [(total_distances <= threshold).float().sum().item()/ num_samples * 100 for threshold in x_values]

            sns.lineplot(x=x_values, y=asr_values, marker=markers[idx % len(markers)], color=color,
                         linestyle=line_styles[idx % len(line_styles)], linewidth=2, markersize=5, alpha=0.8, label=label)

    attack_type = 'adaptive attack' if detector_enabled else 'attack'

    plt.xlabel(f'Distance Budget ({norm} norm)', fontsize=14, labelpad=10)
    plt.ylabel('Attack Success Rate (ASR) (%)', fontsize=14, labelpad=10)
    plt.title(f'ASR (%) vs. Distance under {attack_type}', fontsize=16, pad=10)
    plt.legend(title='Adversarial Sets', fontsize=10, title_fontsize=12)

    # Set x-axis ticks at intervals
    plt.xticks(ticks=range(x_min, x_max+1, interval))

    # Set y-axis ticks for percentage values
    plt.yticks(ticks=range(0, 110, 10))

    plt.grid(True, which='major', linestyle='--', linewidth=0.5)
    plt.minorticks_on()
    plt.grid(True, which='minor', linestyle=':', linewidth=0.5)
    sns.despine()
    plt.tight_layout()
    plt.show()

In [ ]:
import torch
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm

def calculate_distances(adv_batch, mal_x_batch, mask, norm):
    """
    Calculates the distance between adversarial and original samples based on the specified norm.

    Parameters:
    - adv_batch (torch.Tensor): Adversarial samples batch.
    - mal_x_batch (torch.Tensor): Original samples batch.
    - mask (torch.Tensor): Boolean mask for successful attacks.
    - norm (str): The norm type ('L0', 'L1', 'L2').

    Returns:
    - torch.Tensor: The calculated distances.
    """
    adv_samples = adv_batch[mask]
    original_samples = mal_x_batch[mask]

    if norm == 'L0':
        return (adv_samples != original_samples).float().sum(dim=1)
    elif norm == 'L1':
        return torch.abs(adv_samples - original_samples).sum(dim=1)
    elif norm == 'L2':
        return torch.norm(adv_samples - original_samples, p=2, dim=1)
    else:
        raise ValueError("Unsupported norm. Use 'L0', 'L1', or 'L2'.")

def plot_asr_vs_distance(
    adv_rounded_list, labels, models, mal_x_list,
    detector_enableds, device='cpu', batch_size=512,
    norm='L0', interval=5, save_path='asr_vs_distance.eps'
):
    """
    Plots ASR vs. Distance and saves the plot in EPS format for LaTeX documents.

    Parameters:
    - adv_rounded_list (list of torch.Tensor): Adversarial samples for each model.
    - labels (list of str): Labels for each dataset.
    - models (list of models): Models to evaluate.
    - mal_x_list (list of torch.Tensor): Original samples.
    - detector_enabled (bool): Whether the detector is enabled.
    - device (str): Device for computations ('cpu' or 'cuda').
    - batch_size (int): Batch size for processing.
    - norm (str): Norm type ('L0', 'L1', 'L2').
    - interval (int): Tick interval for the x-axis.
    - save_path (str): Path to save the plot in EPS format.
    """
    sns.set(style='whitegrid')
    colors = sns.color_palette('husl', len(adv_rounded_list))
    line_styles = ['-', '--', '-.', ':']
    markers = ['o', 's', '^', 'D', 'v']

    # Enable LaTeX rendering for labels
    plt.rc('text', usetex=True)
    plt.rc('font', family='serif')
    plt.rcParams['text.usetex'] = False

    plt.figure(figsize=(9, 6))
    x_min, x_max = 0, 0

    for idx, (model, adv_rounded, mal_x, detector_enabled, color, label) in tqdm(
        enumerate(zip(models, adv_rounded_list, mal_x_list, detector_enableds, colors, labels)),
        total=len(adv_rounded_list), desc="Processing Tensors"
    ):
        model.eval()
        adv_rounded = adv_rounded.to(device)
        num_samples = adv_rounded.size(0)
        all_distances = []

        for start in range(0, num_samples, batch_size):
            end = min(start + batch_size, num_samples)
            adv_batch = adv_rounded[start:end]
            mal_x_batch = mal_x[start:end]

            with torch.no_grad():
                if detector_enabled:
                    if model.name == 'kde':
                        logits, x_hidden = model.forward_f(adv_batch)
                        prob = model.forward_g(x_hidden, y_pred=0)
                    else:
                        logits, prob = model.forward(adv_batch)
                    successful_attacks_mask = (logits.argmax(dim=1) == 0) & (prob <= model.tau[0].item())
                else:
                    logits = model.forward(adv_batch)
                    successful_attacks_mask = (logits.argmax(dim=1) == 0)

            if successful_attacks_mask.any():
                distances = calculate_distances(adv_batch, mal_x_batch, successful_attacks_mask, norm)
                all_distances.append(distances)

        if all_distances:
            total_distances = torch.cat(all_distances).cpu()
            x_max = max(x_max, int(total_distances.max().item()))
            x_values = np.arange(x_max)

            # Calculate ASR for each distance threshold
            asr_values = [(total_distances <= threshold).float().sum().item() / num_samples * 100 for threshold in x_values]

            sns.lineplot(
                x=x_values, y=asr_values, marker=markers[idx % len(markers)],
                color=color, linestyle=line_styles[idx % len(line_styles)],
                linewidth=2, markersize=5, alpha=1., label=label
            )


    plt.xlabel(f'k', fontsize=14, labelpad=10)
    plt.ylabel('ASR (%)', fontsize=14, labelpad=10)
    #plt.title(f'ASR (%) vs. Distance under attack', fontsize=16, pad=10)
    plt.legend(title='Adversarial Defence', fontsize=9, title_fontsize=10)
    #plt.legend(loc='upper left')

    # Set x-axis ticks at intervals
    plt.xticks(ticks=range(x_min, x_max + 1, interval))

    # Set y-axis ticks for percentage values
    plt.yticks(ticks=range(0, 110, 10))

    plt.grid(True, which='major', linestyle='--', linewidth=0.4)
    plt.minorticks_on()
    plt.grid(True, which='minor', linestyle=':', linewidth=0.4)
    sns.despine()
    plt.tight_layout()

    # Save the plot in EPS format
    try:
        plt.savefig(save_path, format='eps', dpi=600)
        print(f"Plot successfully saved to {save_path}.")
    except Exception as e:
        print(f"Error saving plot: {e}")

    # Show the plot
    plt.show()

In [ ]:
import torch
import gzip
import zipfile
import os


def save_model_data_as_zip(
    model_name,
    output_dir="output",
    **tensors,
):
    """
    Save model data tensors as compressed binary files and package them into a zip file.

    Args:
        model_name (str): Name of the model (e.g., "ICNN").
        output_dir (str): Directory to save the compressed files and the zip.
        detector_enabled (bool): Whether the detector is enabled.
        indicator_masking (bool): Whether indicator masking is enabled.
        **tensors: Keyword arguments for tensors (e.g., mals, advs, mals_detect, etc.).
                   Provide tensors dynamically depending on the configuration.

    Required Tensors:
        - If `detector_enabled` is False:
            * mals
            * advs

    """
    # Ensure the output directory exists
    os.makedirs(output_dir, exist_ok=True)

    # Prepare the tensor dictionary based on configurations

    required_keys = ["mals", "advs"]

    # Check that all required tensors are provided
    missing_keys = [key for key in required_keys if key not in tensors]
    if missing_keys:
        raise ValueError(f"Missing required tensors: {missing_keys}")

    # Generate file paths and tensors to save
    file_prefix = model_name
    tensors_to_save = {}

    tensors_to_save.update({
        f"{file_prefix}_mals_sigma.pt.gz": tensors["mals"],
        f"{file_prefix}_advs_sigma.pt.gz": tensors["advs"],
    })

    # Save tensors as compressed .pt.gz files
    for file_name, tensor in tensors_to_save.items():
        file_path = os.path.join(output_dir, file_name)
        with gzip.open(file_path, "wb") as f:
            torch.save(tensor.clone(), f)

    # Package all files into a single zip archive
    zip_file_path = os.path.join(output_dir, f"{model_name}_data_sigma.zip")
    with zipfile.ZipFile(zip_file_path, "w") as zipf:
        for file_name in tensors_to_save.keys():
            file_path = os.path.join(output_dir, file_name)
            zipf.write(file_path, arcname=file_name)

    print(f"All files saved and zipped into: {zip_file_path}")

In [ ]:
import torch
import gzip
import zipfile
import os

def load_model_data_from_zip(zip_file_path, output_dir="/content/data"):
    """
    Load tensors from a zipped archive containing compressed .pt.gz files and convert them to float.

    Args:
        zip_file_path (str): Path to the zipped archive (e.g., "model_name_data.zip").
        output_dir (str): Directory to extract the files temporarily.

    Returns:
        dict: A dictionary where keys are file names (without extensions) and values are the loaded tensors as float.
    """
    # Ensure the output directory exists
    os.makedirs(output_dir, exist_ok=True)

    # Extract the zip file
    with zipfile.ZipFile(zip_file_path, "r") as zipf:
        zipf.extractall(output_dir)
        print(f"Extracted files to {output_dir}")

    # Load tensors from extracted .pt.gz files and convert to float
    tensors = {}
    for file_name in os.listdir(output_dir):
        if file_name.endswith(".pt.gz"):
            file_path = os.path.join(output_dir, file_name)
            with gzip.open(file_path, "rb") as f:
                tensor_name = os.path.splitext(os.path.splitext(file_name)[0])[0]  # Remove .pt.gz
                tensors[tensor_name] = torch.load(f, weights_only=False).to(dtype=torch.float)  # Convert to float
                print(f"Loaded and converted tensor: {tensor_name}")

    return tensors

### Step 3 — DNN

# Evaluations

In [ ]:
for x_test, y_test in test_loader:
      x_test, y_test = x_test.to(device), y_test.to(device)
      mal_x_batch, mal_y_batch = x_test[y_test.squeeze() == 1], y_test[y_test.squeeze() == 1]
      ben_x_batch, ben_y_batch = x_test[y_test.squeeze() == 0], y_test[y_test.squeeze() == 0]
      print(ben_x_batch.shape)
      print(mal_x_batch.shape)
      break

# Set a random seed for reproducibility
torch.manual_seed(42)  # Replace 42 with your desired seed

# Randomly select 1000 indices
num_samples_to_select = 800
indices = torch.randperm(mal_x_batch.size(0))[:num_samples_to_select]

# Get the subset of the tensor
mal_x = mal_x_batch[indices]

print(mal_x.mean())
print(f"Selected tensor shape: {mal_x.shape}")

## DNN

In [ ]:
model_dnn = MalwareDetectionDNN(10000,2,device=device, name='DNN')
model_dnn.load('/content/model_dnn.pth')
model_dnn.to(device).float()

In [ ]:
lrs = [0.4,0.5,0.6]
thresholds = [0.2,0.3,0.4,0.5]
for lr in lrs:
    for threshold in thresholds:
        print(f"{'-'*50}")
        print(f'learning rate : {lr} and  threshold : {threshold}')
        advs_adapt = sigma_zero(
            model=model_dnn,
            original_inputs=mal_x,
            feature_mask=feature_mask,
            max_iterations=10000,
            learning_rate=lr,
            sigma=1e-4,
            threshold=threshold,
            verbose=True,
            grad_norm=torch.inf,
            device=device,
            detector_enabled=False,
        )

In [ ]:
lrs = [0.5,0.6]
thresholds = [0.1,0.2,0.3,0.4]
for lr in lrs:
    for threshold in thresholds:
        print(f"{'-'*50}")
        print(f'learning rate : {lr} and  threshold : {threshold}')
        advs_adapt = sigma_zero(
            model=model_dnn,
            original_inputs=mal_x,
            feature_mask=feature_mask,
            max_iterations=10000,
            learning_rate=lr,
            sigma=1e-4,
            threshold=threshold,
            verbose=True,
            grad_norm=torch.inf,
            device=device,
            detector_enabled=False,
        )

In [ ]:
mals_adapt, advs_adapt = attack_mal_batches(
    test_loader=test_loader,
    model=model_dnn,
    attack=sigma_zero,
    feature_mask=feature_mask,
    device=device,
    detector_enabled=False,
    max_iterations=10000,
    learning_rate=0.5,
    sigma=1e-4,
    threshold=0.1,
    verbose=True,
    grad_norm=torch.inf,
)

In [ ]:
ensure_fixed_elements_unchanged(mals_adapt, advs_adapt, feature_mask)

In [ ]:
_ , _ = evaluate_adversarial_examples(mal_x=mals_adapt, adv=advs_adapt, model=model_dnn, feature_mask=feature_mask,
                                      detector_enabled=False, oblivion=False, indicator_masking=False, rounded=False, batch_size=batch_size, norm='L0')

_ , _ = evaluate_adversarial_examples(mal_x=mals_adapt, adv=advs_adapt, model=model_dnn, feature_mask=feature_mask,
                                      detector_enabled=False, oblivion=False, indicator_masking=False, rounded=True, batch_size=batch_size, norm='L0')

In [ ]:
model_name = model_dnn.name

save_model_data_as_zip(
    model_name=model_name,
    output_dir="/content/",
    mals=mals_adapt,
    advs=advs_adapt,
)

In [ ]:
DNN_tensors = load_model_data_from_zip(zip_file_path="/content/DNN_data_sigma.zip", output_dir="/content/data")

advs_adapt_DNN , mals_adapt_DNN = evaluate_adversarial_examples(mal_x=DNN_tensors['DNN_mals_sigma'], adv=DNN_tensors['DNN_advs_sigma'], model=model_dnn, feature_mask=feature_mask,
                                      detector_enabled=False, oblivion=False, indicator_masking=False, rounded=False, batch_size=batch_size, norm='L0')

advs_adapt_DNN , mals_adapt_DNN = evaluate_adversarial_examples(mal_x=DNN_tensors['DNN_mals_sigma'], adv=DNN_tensors['DNN_advs_sigma'], model=model_dnn, feature_mask=feature_mask,
                                      detector_enabled=False, oblivion=False, indicator_masking=False, rounded=True, batch_size=batch_size, norm='L0')

### Step 4 — AT-rFGSM

## rFGSM

In [ ]:
model_rFGSM = MalwareDetectionDNN(10000,2,device=device, name='rFGSM')
model_rFGSM.load2('/content/model_at_pgd.pth')
model_rFGSM.to(device).float()

In [ ]:
lrs = [0.7,0.8,0.9]
thresholds = [0.4,0.5,0.6,0.7]
for lr in lrs:
    for threshold in thresholds:
        print(f"{'-'*50}")
        print(f'learning rate : {lr} and  threshold : {threshold}')
        advs_adapt = sigma_zero(
            model=model_rFGSM,
            original_inputs=mal_x,
            feature_mask=feature_mask,
            max_iterations=10000,
            learning_rate=lr,
            sigma=1e-4,
            threshold=threshold,
            verbose=True,
            grad_norm=torch.inf,
            device=device,
            detector_enabled=False,
        )

In [ ]:
mals_adapt, advs_adapt = attack_mal_batches(
    test_loader=test_loader,
    model=model_rFGSM,
    attack=sigma_zero,
    feature_mask=feature_mask,
    device=device,
    detector_enabled=False,
    max_iterations=10000,
    learning_rate=0.8,
    sigma=1e-4,
    threshold=0.5,
    verbose=True,
    grad_norm=torch.inf,
)

In [ ]:
_ , _ = evaluate_adversarial_examples(mal_x=mals_adapt, adv=advs_adapt, model=model_rFGSM, feature_mask=feature_mask,
                                      detector_enabled=False, oblivion=False, indicator_masking=False, rounded=False, batch_size=batch_size, norm='L0')

_ , _ = evaluate_adversarial_examples(mal_x=mals_adapt, adv=advs_adapt, model=model_rFGSM, feature_mask=feature_mask,
                                      detector_enabled=False, oblivion=False, indicator_masking=False, rounded=True, batch_size=batch_size, norm='L0')

In [ ]:
model_name = model_rFGSM.name

save_model_data_as_zip(
    model_name=model_name,
    output_dir="/content/",
    mals=mals_adapt,
    advs=advs_adapt,
)

In [ ]:
rFGSM_tensors = load_model_data_from_zip(zip_file_path="/content/rFGSM_data_sigma.zip", output_dir="/content/data")

advs_adapt_rFGSM , mals_adapt_rFGSM = evaluate_adversarial_examples(mal_x=rFGSM_tensors['rFGSM_mals_sigma'], adv=rFGSM_tensors['rFGSM_advs_sigma'], model=model_rFGSM, feature_mask=feature_mask,
                                      detector_enabled=False, oblivion=False, indicator_masking=False, rounded=False, batch_size=batch_size, norm='L0')

advs_adapt_rFGSM , mals_adapt_rFGSM = evaluate_adversarial_examples(mal_x=rFGSM_tensors['rFGSM_mals_sigma'], adv=rFGSM_tensors['rFGSM_advs_sigma'], model=model_rFGSM, feature_mask=feature_mask,
                                      detector_enabled=False, oblivion=False, indicator_masking=False, rounded=True, batch_size=batch_size, norm='L0')

### Step 5 — AT-MaxMA

## MaxMA

In [ ]:
model_MaxMA = MalwareDetectionDNN(10000,2,device=device, name='MaxMA')
model_MaxMA.load2('/content/model_at_ma.pth')
model_MaxMA.to(device).float()

In [ ]:
lrs = [3.8,4.,4.2,4.4]
thresholds = [0.3,0.4,0.5]
for lr in lrs:
    for threshold in thresholds:
        print(f"{'-'*50}")
        print(f'learning rate : {lr} and  threshold : {threshold}')
        advs_adapt = sigma_zero(
            model=model_MaxMA,
            original_inputs=mal_x,
            feature_mask=feature_mask,
            max_iterations=10000,
            learning_rate=lr,
            sigma=1e-4,
            threshold=threshold,
            verbose=True,
            grad_norm=torch.inf,
            device=device,
            detector_enabled=False,
        )

In [ ]:
lrs = [3.8,3.6,3.4,3.2,3.]
thresholds = [0.4,0.5,0.6]
for lr in lrs:
    for threshold in thresholds:
        print(f"{'-'*50}")
        print(f'learning rate : {lr} and  threshold : {threshold}')
        advs_adapt = sigma_zero(
            model=model_MaxMA,
            original_inputs=mal_x,
            feature_mask=feature_mask,
            max_iterations=10000,
            learning_rate=lr,
            sigma=1e-4,
            threshold=threshold,
            verbose=True,
            grad_norm=torch.inf,
            device=device,
            detector_enabled=False,
        )

In [ ]:
mals_adapt, advs_adapt = attack_mal_batches(
    test_loader=test_loader,
    model=model_MaxMA,
    attack=sigma_zero,
    feature_mask=feature_mask,
    device=device,
    detector_enabled=False,
    max_iterations=10000,
    learning_rate=3.8,
    sigma=1e-4,
    threshold=0.5,
    verbose=True,
    grad_norm=torch.inf,
)

In [ ]:
_ , _ = evaluate_adversarial_examples(mal_x=mals_adapt, adv=advs_adapt, model=model_MaxMA, feature_mask=feature_mask,
                                      detector_enabled=False, oblivion=False, indicator_masking=False, rounded=False, batch_size=batch_size, norm='L0')

_ , _ = evaluate_adversarial_examples(mal_x=mals_adapt, adv=advs_adapt, model=model_MaxMA, feature_mask=feature_mask,
                                      detector_enabled=False, oblivion=False, indicator_masking=False, rounded=True, batch_size=batch_size, norm='L0')

In [ ]:
model_name = model_MaxMA.name

save_model_data_as_zip(
    model_name=model_name,
    output_dir="/content/",
    mals=mals_adapt,
    advs=advs_adapt,
)

In [ ]:
MaxMA_tensors = load_model_data_from_zip(zip_file_path="/content/MaxMA_data_sigma.zip", output_dir="/content/data")

advs_adapt_MaxMA , mals_adapt_MaxMA = evaluate_adversarial_examples(mal_x=MaxMA_tensors['MaxMA_mals_sigma'], adv=MaxMA_tensors['MaxMA_advs_sigma'], model=model_MaxMA, feature_mask=feature_mask,
                                      detector_enabled=False, oblivion=False, indicator_masking=False, rounded=False, batch_size=batch_size, norm='L0')

advs_adapt_MaxMA , mals_adapt_MaxMA = evaluate_adversarial_examples(mal_x=MaxMA_tensors['MaxMA_mals_sigma'], adv=MaxMA_tensors['MaxMA_advs_sigma'], model=model_MaxMA, feature_mask=feature_mask,
                                      detector_enabled=False, oblivion=False, indicator_masking=False, rounded=True, batch_size=batch_size, norm='L0')

### Step 6 — DLA

## DLA

In [ ]:
model_dnn = MalwareDetectionDNN(10000,2,device=device, name='DNN')
model_dnn.load('/content/model_dnn.pth')

model_dla = AMalwareDetectionDLA(model_dnn,input_size=10000,n_classes=2,device=device,name='DLA')
model_dla.load('/content/amd_dla.pth')
model_dla.to(device).float()

In [ ]:
lrs = [0.4,0.5,0.6]
thresholds = [0.1,0.2,0.3,0.4]
for lr in lrs:
    for threshold in thresholds:
        print(f"{'-'*50}")
        print(f'learning rate : {lr} and  threshold : {threshold}')
        advs_adapt = sigma_zero(
            model=model_dla,
            original_inputs=mal_x,
            feature_mask=feature_mask,
            max_iterations=10000,
            learning_rate=lr,
            sigma=1e-4,
            threshold=threshold,
            verbose=True,
            grad_norm=torch.inf,
            device=device,
            detector_enabled=True,
            oblivion= False,
            binary_search_steps= 4,
            initial_const=0.1,
        )

In [ ]:
mals_adapt, advs_adapt = attack_mal_batches(
    test_loader=test_loader,
    model=model_dla,
    attack=sigma_zero,
    feature_mask=feature_mask,
    device=device,
    detector_enabled=False,
    max_iterations=10000,
    learning_rate=3.8,
    sigma=1e-4,
    threshold=0.5,
    verbose=True,
    grad_norm=torch.inf,
)

In [ ]:
mals_adapt, advs_adapt = attack_mal_batches(
    test_loader=test_loader,
    model=model_dla,
    attack=sigma_zero_rounded_alpha,
    feature_mask=feature_mask,
    device=device,
    detector_enabled=True,
    oblivion=False,
    indicator_masking=False,
    max_iterations=10000,
    learning_rate=0.5,
    sigma=1e-4,
    threshold=0.2,
    verbose=True,
    grad_norm=torch.inf,
    confidence_bound_primary = 0.4,
    confidence_bound_secondary = 0.4,
    confidence_update_interval = 80,
    binary_search_steps = 4,
    initial_const = 0.1,
)

In [ ]:
_ , _ = evaluate_adversarial_examples(mal_x=mals_adapt, adv=advs_adapt, model=model_dla, feature_mask=feature_mask,
                                      detector_enabled=True, oblivion=False, indicator_masking=False, rounded=False, batch_size=batch_size, norm='L0')

_ , _ = evaluate_adversarial_examples(mal_x=mals_adapt, adv=advs_adapt, model=model_dla, feature_mask=feature_mask,
                                      detector_enabled=True, oblivion=False, indicator_masking=False, rounded=True, batch_size=batch_size, norm='L0')

In [ ]:
model_name = model_dla.name

save_model_data_as_zip(
    model_name=model_name,
    output_dir="/content/",
    mals=mals_adapt,
    advs=advs_adapt,
)

### Step 7 — PAD

## PAD

In [ ]:
model_dnn = MalwareDetectionDNN(10000,2,device=device, name='DNN')
model_dnn.load('/content/model_dnn.pth')

model_pad = AdvMalwareDetectorICNN(model_dnn,input_size=10000,n_classes=2,device=device,name='PAD')
model_pad.load2('/content/amd_pad_ma.pth')
model_pad.to(device).float()

In [ ]:
lrs = [0.25,0.3,0.35]
thresholds = [0.2,0.3,0.4]
for lr in lrs:
    for threshold in thresholds:
        print(f"{'-'*50}")
        print(f'learning rate : {lr} and  threshold : {threshold}')
        advs_adapt = sigma_zero(
            model=model_pad,
            original_inputs=mal_x,
            feature_mask=feature_mask,
            max_iterations=10000,
            learning_rate=lr,
            sigma=1e-4,
            threshold=threshold,
            verbose=True,
            grad_norm=torch.inf,
            device=device,
            detector_enabled=True,
            oblivion= False,
            binary_search_steps= 4,
            initial_const= 0.1
        )

In [ ]:
mals_adapt, advs_adapt = attack_mal_batches(
    test_loader=test_loader,
    model=model_pad,
    attack=sigma_zero,
    feature_mask=feature_mask,
    device=device,
    detector_enabled=True,
    oblivion=False,
    indicator_masking=False,
    max_iterations=10000,
    learning_rate=0.3,
    sigma=1e-4,
    threshold=0.3,
    verbose=True,
    grad_norm=torch.inf,
    binary_search_steps = 4,
    initial_const = 0.1,
)

In [ ]:
_ , _ = evaluate_adversarial_examples(mal_x=mals_adapt, adv=advs_adapt, model=model_pad, feature_mask=feature_mask,
                                      detector_enabled=True, oblivion=False, indicator_masking=False, rounded=False, batch_size=batch_size, norm='L0')

_ , _ = evaluate_adversarial_examples(mal_x=mals_adapt, adv=advs_adapt, model=model_pad, feature_mask=feature_mask,
                                      detector_enabled=True, oblivion=False, indicator_masking=False, rounded=True, batch_size=batch_size, norm='L0')

In [ ]:
model_name = model_pad.name

save_model_data_as_zip(
    model_name=model_name,
    output_dir="/content/",
    mals=mals_adapt,
    advs=advs_adapt,
)

In [ ]:
PAD_tensors = load_model_data_from_zip(zip_file_path="/content/PAD_data_sigma.zip", output_dir="/content/data")

advs_adapt_PAD , mals_adapt_PAD = evaluate_adversarial_examples(mal_x=PAD_tensors['PAD_mals_sigma'], adv=PAD_tensors['PAD_advs_sigma'], model=model_pad, feature_mask=feature_mask,
                                      detector_enabled=True, oblivion=False, indicator_masking=False, rounded=False, batch_size=batch_size, norm='L0')

advs_adapt_PAD , mals_adapt_PAD = evaluate_adversarial_examples(mal_x=PAD_tensors['PAD_mals_sigma'], adv=PAD_tensors['PAD_advs_sigma'], model=model_pad, feature_mask=feature_mask,
                                      detector_enabled=True, oblivion=False, indicator_masking=False, rounded=True, batch_size=batch_size, norm='L0')

## Part B — Tuning σ for the detectors (from `parameter_tuning.ipynb`)

### Step 8 — KDE

In [ ]:
for x_test, y_test in test_loader:
      x_test, y_test = x_test.to(device), y_test.to(device)
      mal_x_batch, mal_y_batch = x_test[y_test.squeeze() == 1], y_test[y_test.squeeze() == 1]
      ben_x_batch, ben_y_batch = x_test[y_test.squeeze() == 0], y_test[y_test.squeeze() == 0]
      print(ben_x_batch.shape)
      print(mal_x_batch.shape)
      break

# Set a random seed for reproducibility
torch.manual_seed(42)  # Replace 42 with your desired seed

# Randomly select 1000 indices
num_samples_to_select = 1000
indices = torch.randperm(mal_x_batch.size(0))[:num_samples_to_select]

# Get the subset of the tensor
mal_x = mal_x_batch[indices]

print(mal_x.mean())
print(f"Selected tensor shape: {mal_x.shape}")

# KDE

In [ ]:
model_dnn = MalwareDetectionDNN(10000,2,device=device, name='DNN')
model_dnn.load('/content/model_dnn.pth')

model_kde = KernelDensityEstimation(model_dnn,n_centers=1000,bandwidth=16.,n_classes=2,ratio=0.95,device=device,name='KDE')
model_kde.load('/content/amd_kde.pth')
model_kde.to(device).float()

In [ ]:
lrs = [0.4,0.5,0.6]
thresholds = [0.2,0.3,0.4,0.5]
for lr in lrs:
    for threshold in thresholds:
        advs_rounded = sigma_zero_rounded(model=model_kde, original_inputs=mal_x, feature_mask=feature_mask, max_iterations=2000,
                                          learning_rate=lr, sigma=1e-4, threshold=threshold, verbose=True, grad_norm=torch.inf, detector_enabled=True,
                                          oblivion=False, binary_search_steps=4, initial_const=0.1, device=device)

### Step 9 — DLA, including the γ/α grid

# DLA

In [ ]:
lrs = [0.5,0.6,0.7,0.8]
thresholds = [0.2,0.3,0.4,0.5]
for lr in lrs:
    for threshold in thresholds:
        print(f"{'-'*50}")
        print(f'learning rate : {lr} and  threshold : {threshold}')

        advs_rounded = sigma_zero_rounded(model=model_dla, original_inputs=mal_x, feature_mask=feature_mask, max_iterations=2000,
                                          learning_rate=lr, sigma=1e-4, threshold=threshold, verbose=True, grad_norm=torch.inf, detector_enabled=True,
                                          oblivion=False, binary_search_steps=4, initial_const=0.1, device=device)

In [ ]:
lrs = [0.3,0.4]
thresholds = [0.1,0.2,0.3,0.4,0.5]
for lr in lrs:
    for threshold in thresholds:
        print(f"{'-'*50}")
        print(f'learning rate : {lr} and  threshold : {threshold}')

        advs_rounded = sigma_zero_rounded(model=model_dla, original_inputs=mal_x, feature_mask=feature_mask, max_iterations=2000,
                                          learning_rate=lr, sigma=1e-4, threshold=threshold, verbose=True, grad_norm=torch.inf, detector_enabled=True,
                                          oblivion=False, binary_search_steps=4, initial_const=0.1, device=device)

In [ ]:
lrs = [0.5]
thresholds = [0.1]
for lr in lrs:
    for threshold in thresholds:
        print(f"{'-'*50}")
        print(f'learning rate : {lr} and  threshold : {threshold}')

        advs_rounded = sigma_zero_rounded(model=model_dla, original_inputs=mal_x, feature_mask=feature_mask, max_iterations=2000,
                                          learning_rate=lr, sigma=1e-4, threshold=threshold, verbose=True, grad_norm=torch.inf, detector_enabled=True,
                                          oblivion=False, binary_search_steps=4, initial_const=0.1, device=device)

In [ ]:
gammas = [40, 60, 80]
alphas = [0.4, 0.8, 1.2]
for gamma in gammas:
    for alpha in alphas:
        print(f"{'-'*50}")
        print(f'gamma : {gamma} and  alpha : {alpha}')

        advs_rounded = sigma_zero_rounded_alpha(model=model_dla, original_inputs=mal_x, feature_mask=feature_mask, max_iterations=10000,
                                          learning_rate=0.5, sigma=1e-4, threshold=0.2, verbose=True, grad_norm=torch.inf, detector_enabled=True,
                                          oblivion=False, binary_search_steps=4, initial_const=0.1, confidence_bound_primary=alpha,
                                          confidence_bound_secondary=alpha, confidence_update_interval=gamma, device=device)

In [ ]:
gammas = [60, 80]
alphas = [0.4, 0.8, 1.2]
for gamma in gammas:
    for alpha in alphas:
        print(f"{'-'*50}")
        print(f'gamma : {gamma} and  alpha : {alpha}')

        advs_rounded = sigma_zero_rounded_alpha(model=model_dla, original_inputs=mal_x, feature_mask=feature_mask, max_iterations=10000,
                                          learning_rate=0.5, sigma=1e-4, threshold=0.2, verbose=True, grad_norm=torch.inf, detector_enabled=True,
                                          oblivion=False, binary_search_steps=4, initial_const=0.1, confidence_bound_primary=alpha,
                                          confidence_bound_secondary=alpha, confidence_update_interval=gamma, device=device)

### Step 10 — DNN+

# DNN_Plus

In [ ]:
model_dnn = MalwareDetectionDNN(10000,2,device=device, name='DNN')
model_dnn.load('/content/model_dnn.pth')

model_dnn_plus = AMalwareDetectionDNNPlus(model_dnn,input_size=10000,n_classes=2,device=device,name='DNNPlus')
model_dnn_plus.load('/content/amd_dnn_plus.pth')
model_dnn_plus.to(device).float()

In [ ]:
lrs = [0.5,0.6,0.7,0.8]
thresholds = [0.2,0.3,0.4,0.5]
for lr in lrs:
    for threshold in thresholds:
        print(f"{'-'*50}")
        print(f'learning rate : {lr} and  threshold : {threshold}')

        advs_rounded = sigma_zero_rounded(model=model_dnn_plus, original_inputs=mal_x, feature_mask=feature_mask, max_iterations=2000,
                                          learning_rate=lr, sigma=1e-4, threshold=threshold, verbose=True, grad_norm=torch.inf, detector_enabled=True,
                                          oblivion=False, binary_search_steps=4, initial_const=0.1, device=device)

### Step 11 — ICNN

# ICNN

In [ ]:
model_dnn = MalwareDetectionDNN(10000,2,device=device, name='DNN')
model_dnn.load('/content/model_dnn.pth')

model_icnn = AdvMalwareDetectorICNN(model_dnn,input_size=10000,n_classes=2,device=device,name='ICNN')
model_icnn.load('/content/amd_icnn.pth')
model_icnn.to(device).float()

In [ ]:
lrs = [0.5,0.6,0.7,0.8]
thresholds = [0.2,0.3,0.4,0.5]
for lr in lrs:
    for threshold in thresholds:
        print(f"{'-'*50}")
        print(f'learning rate : {lr} and  threshold : {threshold}')

        advs_rounded = sigma_zero_rounded(model=model_icnn, original_inputs=mal_x, feature_mask=feature_mask, max_iterations=2000,
                                          learning_rate=lr, sigma=1e-4, threshold=threshold, verbose=True, grad_norm=torch.inf, detector_enabled=True,
                                          oblivion=False, binary_search_steps=4, initial_const=0.1, device=device)

In [ ]:
lrs = [0.6,0.7,0.8]
thresholds = [0.2,0.3,0.4,0.5]
for lr in lrs:
    for threshold in thresholds:
        print(f"{'-'*50}")
        print(f'learning rate : {lr} and  threshold : {threshold}')

        advs_rounded = sigma_zero_rounded(model=model_icnn, original_inputs=mal_x, feature_mask=feature_mask, max_iterations=2000,
                                          learning_rate=lr, sigma=1e-4, threshold=threshold, verbose=True, grad_norm=torch.inf, detector_enabled=True,
                                          oblivion=False, binary_search_steps=4, initial_const=0.1, device=device)

In [ ]:
lrs = [0.5]
thresholds = [0.2,0.3,0.4,0.5]
for lr in lrs:
    for threshold in thresholds:
        print(f"{'-'*50}")
        print(f'learning rate : {lr} and  threshold : {threshold}')

        advs_rounded = sigma_zero_rounded(model=model_icnn, original_inputs=mal_x, feature_mask=feature_mask, max_iterations=2000,
                                          learning_rate=lr, sigma=1e-4, threshold=threshold, verbose=True, grad_norm=torch.inf, detector_enabled=True,
                                          oblivion=False, binary_search_steps=4, initial_const=0.1, device=device)